In [ ]:
import json
import os
import pandas as pd
import ijson
import pandas as pd
from itertools import islice
import sys
import numpy as np
import scipy


pd.set_option('display.max_rows', 100)

In [4]:


# Define the path where your Visual Genome dataset files are located
dataset_path = "vg"

# List of files to check
data_files = [
    "scene_graphs.json",
    "objects.json", 
    "relationships.json", 
    "attributes.json",  
    "region_descriptions.json", 
    "object_synsets.json"
]

In [14]:
def print_first_n_chars_of_json(file_path, n):
    with open(file_path, 'r') as file:
        # Read the first n characters
        content = file.read(n)
        print(content)

print_first_n_chars_of_json('vg/scene_graphs.json', 11000)

[{"relationships": [{"synsets": ["along.r.01"], "predicate": "ON", "relationship_id": 15927, "object_id": 5046, "subject_id": 5045}, {"synsets": ["wear.v.01"], "predicate": "wears", "relationship_id": 15928, "object_id": 5048, "subject_id": 1058529}, {"synsets": ["have.v.01"], "predicate": "has", "relationship_id": 15929, "object_id": 5050, "subject_id": 5049}, {"synsets": ["along.r.01"], "predicate": "ON", "relationship_id": 15930, "object_id": 1058508, "subject_id": 1058507}, {"synsets": ["along.r.01"], "predicate": "ON", "relationship_id": 15931, "object_id": 1058534, "subject_id": 5055}, {"synsets": ["have.v.01"], "predicate": "has", "relationship_id": 15932, "object_id": 1058511, "subject_id": 1058529}, {"synsets": ["next.r.01"], "predicate": "next to", "relationship_id": 15933, "object_id": 1058539, "subject_id": 1058534}, {"synsets": ["have.v.01"], "predicate": "has", "relationship_id": 15934, "object_id": 5060, "subject_id": 1058515}, {"synsets": ["have.v.01"], "predicate": "ha

In [17]:
# Define the number of entries to load
LIMIT = 5

# Load and store the first 100 records from each file
dataframes = {}
for file in data_files:
    print(file)
    file_path = os.path.join(dataset_path, file)
    data = []
    with open(file_path, 'rb') as file:
        # Create a parser for items in the root array
        parser = ijson.items(file, 'item')
        
        # Take only the first num_entries
        for entry in islice(parser, LIMIT):
            data.append(entry)
    #df = []
    print(data[:1])
    # df = pd.DataFrame(data)     
      
    # if df is not None:
    #     dataframes = df
    #     #print(f"Loaded {file} with {df.shape[0]} rows (showing first {LIMIT} instances).")

NameError: name 'data_files' is not defined

In [65]:
LIMIT = 10000
scene_graphs = []
file_path = os.path.join("vg/scene_graphs.json")

with open(file_path, 'rb') as file:
    # Create a parser for items in the root array
    parser = ijson.items(file, 'item')
    
    # Take only the first num_entries
    for entry in islice(parser, LIMIT):
        scene_graphs.append(entry)

In [8]:
print(scene_graphs[0])

{'relationships': [{'synsets': ['along.r.01'], 'predicate': 'ON', 'relationship_id': 15927, 'object_id': 5046, 'subject_id': 5045}, {'synsets': ['wear.v.01'], 'predicate': 'wears', 'relationship_id': 15928, 'object_id': 5048, 'subject_id': 1058529}, {'synsets': ['have.v.01'], 'predicate': 'has', 'relationship_id': 15929, 'object_id': 5050, 'subject_id': 5049}, {'synsets': ['along.r.01'], 'predicate': 'ON', 'relationship_id': 15930, 'object_id': 1058508, 'subject_id': 1058507}, {'synsets': ['along.r.01'], 'predicate': 'ON', 'relationship_id': 15931, 'object_id': 1058534, 'subject_id': 5055}, {'synsets': ['have.v.01'], 'predicate': 'has', 'relationship_id': 15932, 'object_id': 1058511, 'subject_id': 1058529}, {'synsets': ['next.r.01'], 'predicate': 'next to', 'relationship_id': 15933, 'object_id': 1058539, 'subject_id': 1058534}, {'synsets': ['have.v.01'], 'predicate': 'has', 'relationship_id': 15934, 'object_id': 5060, 'subject_id': 1058515}, {'synsets': ['have.v.01'], 'predicate': 'has

In [ ]:
def process_scene_graphs(scene_graphs):
    # Initialize an empty list to store rows
    rows = []

    # Iterate through each scene graph in the list
    for scene_graph in scene_graphs:
        # Extract image_id for the current scene graph
        image_id = scene_graph['image_id']

        # Extract objects and their bounding boxes into a dictionary
        objects_dict = {
            obj['object_id']: {
                'name': obj['names'][0] if obj['names'] else None,
                'bbox': (obj['x'], obj['y'], obj['w'], obj['h']),
                'attributes': obj.get('attributes', [])
            }
            for obj in scene_graph['objects']
        }

        # Add rows for relationships
        for rel in scene_graph['relationships']:
            subject_id, object_id = rel['subject_id'], rel['object_id']
            subject = objects_dict.get(subject_id, {})
            obj = objects_dict.get(object_id, {})
            rows.append({
                'image_id': image_id,
                'subject': subject.get('name'),
                'subject_bbox': list(subject.get('bbox')),  # Ensuring bbox is a list
                'relationship': rel['predicate'],
                'object': obj.get('name'),
                'object_bbox': list(obj.get('bbox'))  # Ensuring bbox is a list
            })

        # Add rows for attributes
        for obj_id, obj_data in objects_dict.items():
            for attr in obj_data['attributes']:
                rows.append({
                    'image_id': image_id,
                    'subject': obj_data['name'],
                    'subject_bbox': list(obj_data['bbox']),  # Ensuring bbox is a list
                    'relationship': 'has_attribute',
                    'object': attr,
                    'object_bbox': None  # No bounding box for attributes
                })
    obj_row = rows[0]
    att_row = rows[50]
    
    
    
    # Create a pandas DataFrame
    df = pd.DataFrame(rows)

    return df

df = process_scene_graphs(scene_graphs)

# Display the resulting DataFrame
df.head(20)

In [ ]:
df_sorted = df.sort_values(by='image_id', ascending=True)

# Reset the index if needed (optional)
df_sorted = df_sorted.reset_index(drop=True)


df.to_csv('vg_scene_graphs.csv', index=False, na_rep='NULL')
df.head(-76)

In [ ]:
import nltk
import pandas as pd
import json
from nltk.corpus import wordnet as wn
from collections import Counter, defaultdict
import re
from tqdm import tqdm  # For progress tracking

# Download WordNet if not already downloaded
nltk.download('wordnet')

def format_term(term: str) -> str:
    """
    Converts terms to the required format:
    - First word lowercase
    - Subsequent words capitalized
    - No spaces or symbols between words
    
    Example:
    "toilet_bowl" -> "toiletBowl"
    "random-access memory" -> "randomAccessMemory"
    """
    # Replace common separators with spaces to normalize
    normalized = term.replace('-', ' ').replace('_', ' ').replace('/', ' ')
    
    # Split into words
    words = normalized.split()
    
    # Format: first word lowercase, rest capitalized
    if not words:
        return ""
    
    formatted = words[0].lower()
    for word in words[1:]:
        formatted += word.capitalize()
    
    return formatted


def get_readable_formatted_name(synset_name):
    try:
        # Check if it's a properly formatted synset name
        if isinstance(synset_name, str) and synset_name.count('.') >= 2:
            parts = synset_name.split('.')
            if len(parts) >= 3 and parts[1] in ('n', 'v', 'a', 's', 'r'):
                try:
                    synset = wn.synset(synset_name)
                    return format_term(synset.lemma_names()[0])
                except:
                    pass
        
        # If we get here, either it's not a valid synset name or there was an error
        return format_term(synset_name)
    except Exception as e:
        print(f"Error formatting '{synset_name}': {e}")
        return format_term(str(synset_name))
    
def create_readable_df(filtered_df):
    readable_df = filtered_df.copy()
    
    # Apply formatting to columns
    readable_df['subject'] = readable_df['subject'].apply(get_readable_formatted_name)
    readable_df['object'] = readable_df['object'].apply(get_readable_formatted_name)
    readable_df['relation'] = readable_df['relation'].apply(format_term)
    
    # Drop object_type column if it exists
    if 'object_type' in readable_df.columns:
        readable_df = readable_df.drop(columns=['object_type'])
    
    return readable_df


def extract_synsets_from_scene_graphs(scene_graphs_path, subject_threshold=100, object_threshold=100):
    print("Loading scene graphs...")
    with open(scene_graphs_path, 'r') as f:
        scene_graphs = json.load(f)
    
    # Create dictionaries for mapping and counting
    synset_counter = Counter()
    
    print(f"Processing {len(scene_graphs)} scene graphs...")
    for graph in tqdm(scene_graphs):
        # Process relationships
        if 'relationships' in graph:
            for rel in graph['relationships']:
                if 'synsets' in rel and rel['synsets']:
                    for synset_name in rel['synsets']:
                        if synset_name:  # Skip empty synsets
                            synset_counter[synset_name] += 1
        
        # Process objects
        if 'objects' in graph:
            for obj in graph['objects']:
                if 'synsets' in obj and obj['synsets']:
                    for synset_name in obj['synsets']:
                        if synset_name:  # Skip empty synsets
                            synset_counter[synset_name] += 1
    
    # Find frequent synsets
    frequent_synsets = {synset for synset, count in synset_counter.items() 
                       if count >= min(subject_threshold, object_threshold)}
    
    print(f"Found {len(frequent_synsets)} frequent synsets (threshold: min({subject_threshold}, {object_threshold}))")
    
    return frequent_synsets, synset_counter

def synset_name_to_wn(synset_name):
    """
    Convert a synset name from Visual Genome format to a WordNet synset object.
    Handles multiple formats and performs robust error handling.
    """
    # Case 1: Direct lookup (standard WordNet format like 'dog.n.01')
    try:
        return wn.synset(synset_name)
    except:
        pass
    
    # Case 2: Format with offset (e.g., 'n.01')
    try:
        parts = synset_name.split('.')
        if len(parts) == 3:
            pos = parts[1]
            offset = parts[2]
            
            pos_map = {'n': 'n', 'v': 'v', 'a': 'a', 's': 'a', 'r': 'r'}
            if pos in pos_map:
                return wn.synset_from_pos_and_offset(pos_map[pos], int(offset))
    except:
        pass
    
    # Case 3: Try extracting lemma and POS
    try:
        # For formats like "along.r.01" - try to find matching synsets with lemma and pos
        parts = synset_name.split('.')
        if len(parts) >= 2:
            lemma = parts[0]
            pos = parts[1]
            
            pos_map = {'n': wn.NOUN, 'v': wn.VERB, 'a': wn.ADJ, 's': wn.ADJ_SAT, 'r': wn.ADV}
            if pos in pos_map:
                synsets = wn.synsets(lemma, pos=pos_map[pos])
                if synsets:
                    # If more specific (with offset), try to find exact match
                    if len(parts) >= 3:
                        offset = parts[2]
                        for s in synsets:
                            if s.offset() == int(offset):
                                return s
                    # Otherwise return first matching synset
                    return synsets[0]
    except:
        pass
    
    return None

def extract_wordnet_relationships_by_synset(frequent_synsets):
    data = []
    success_count = 0
    error_count = 0
    
    print("Extracting WordNet relationships...")
    for synset_name in tqdm(frequent_synsets):
        # Try to convert to WordNet synset
        synset = synset_name_to_wn(synset_name)
        
        if not synset:
            error_count += 1
            if error_count <= 5:  # Limit the number of errors printed
                print(f"Warning: Could not convert synset '{synset_name}' to WordNet format")
            continue
        
        success_count += 1
        
        # Extract relationships from this synset
        
        # Add synonyms
        for lemma in synset.lemmas():
            if lemma.name() != synset.lemma_names()[0]:  # Avoid self-referencing
                data.append({
                    "subject": synset.name(),
                    "relation": "synonym",
                    "object": lemma.name(),
                    "object_type": "lemma"  # Note this is a lemma, not a synset
                })
        
        # Hypernyms (is-a relationships)
        for hypernym in synset.hypernyms():
            data.append({
                "subject": synset.name(),
                "relation": "hypernym",
                "object": hypernym.name(),
                "object_type": "synset"
            })
        
        # Hyponyms (for better coverage)
        for hyponym in synset.hyponyms():
            data.append({
                "subject": synset.name(),
                "relation": "hyponym",
                "object": hyponym.name(),
                "object_type": "synset"
            })
        
        # Meronyms (part-of relationships)
        for meronym in synset.part_meronyms():
            data.append({
                "subject": synset.name(),
                "relation": "part_meronym",
                "object": meronym.name(),
                "object_type": "synset"
            })
        
        for meronym in synset.member_meronyms():
            data.append({
                "subject": synset.name(),
                "relation": "member_meronym",
                "object": meronym.name(),
                "object_type": "synset"
            })
        
        # Add substance meronyms (things the subject is made of)
        for meronym in synset.substance_meronyms():
            data.append({
                "subject": synset.name(),
                "relation": "substance_meronym",
                "object": meronym.name(),
                "object_type": "synset"
            })
        
        # Add holonyms (inverse of meronyms - things the subject is part of)
        for holonym in synset.part_holonyms():
            data.append({
                "subject": synset.name(),
                "relation": "part_holonym",
                "object": holonym.name(),
                "object_type": "synset"
            })
        
        # Attributes (for adjectives)
        for attr in synset.attributes():
            data.append({
                "subject": synset.name(),
                "relation": "attribute",
                "object": attr.name(),
                "object_type": "synset"
            })
        
        # Entailments (for verbs)
        for entailment in synset.entailments():
            data.append({
                "subject": synset.name(),
                "relation": "entailment",
                "object": entailment.name(),
                "object_type": "synset"
            })
        
        # Add also_sees (related concepts)
        for also_see in synset.also_sees():
            data.append({
                "subject": synset.name(),
                "relation": "also_see",
                "object": also_see.name(),
                "object_type": "synset"
            })
    
    print(f"Successfully processed {success_count} synsets out of {len(frequent_synsets)}")
    print(f"Failed to process {error_count} synsets")
    
    if not data:
        print("Warning: No relationships were extracted!")
        # Return an empty DataFrame with the expected columns
        return pd.DataFrame(columns=["subject", "relation", "object", "object_type"])
    
    return pd.DataFrame(data)

def filter_and_clean_relationships(df, word_to_synsets, frequent_subject_synsets, frequent_object_synsets, min_subject_freq=50, min_object_freq=10, relation_types=None):
    """
    Filter relationships based on customizable thresholds for subjects and objects.
    
    Parameters:
    - df: DataFrame with relationships
    - word_to_synsets: Dictionary mapping words to synsets
    - frequent_subject_synsets: Set of synsets that are frequent as subjects
    - frequent_object_synsets: Set of synsets that are frequent as objects
    - min_subject_freq: Minimum frequency for subjects
    - min_object_freq: Minimum frequency for objects
    - relation_types: Optional list of relation types to keep
    """
    # Debug: Check DataFrame structure
    print("DataFrame columns:", df.columns.tolist())
    print("DataFrame shape:", df.shape)
    
    if df.empty:
        print("Warning: Empty DataFrame - no filtering performed")
        return df, df
    
    # Remove duplicate relationships
    df = df.drop_duplicates()
    print(f"After removing duplicates: {len(df)} relationships")
    
    # Create a set of all synsets from word_to_synsets for faster lookups
    all_synsets = set()
    for synsets in word_to_synsets.values():
        all_synsets.update(synsets)
    
    print(f"Total unique synsets in Visual Genome: {len(all_synsets)}")
    
    # Filter specific relation types if requested
    if relation_types and len(relation_types) > 0:
        df = df[df['relation'].isin(relation_types)]
        print(f"After filtering for relation types {relation_types}: {len(df)} relationships")
    
    # Function to check if a synset is in frequent subject/object sets
    def is_frequent_subject(synset_name):
        return synset_name in frequent_subject_synsets
    
    def is_frequent_object(synset_name, object_type):
        if object_type == "synset":
            return synset_name in frequent_object_synsets
        elif object_type == "lemma":
            # For lemmas, check if ANY synset with this lemma name is frequent
            # This is more strict and will filter out uncommon lemmas
            lemma_synsets = [s.name() for s in wn.synsets(synset_name)]
            return any(s in frequent_object_synsets for s in lemma_synsets)
        return False  # Default to filtering out
    
    # Filter based on subject frequency
    subject_filtered = df[df['subject'].apply(is_frequent_subject)]
    print(f"After subject frequency filtering (min freq: {min_subject_freq}): {len(subject_filtered)} relationships")
    
    # Filter based on object frequency
    if 'object_type' in subject_filtered.columns:
        final_filtered = subject_filtered[subject_filtered.apply(
            lambda row: is_frequent_object(row['object'], row['object_type']), axis=1)]
    else:
        # Fallback if object_type column is missing
        final_filtered = subject_filtered[subject_filtered['object'].apply(
            lambda x: x in frequent_object_synsets if '.' in x and len(x.split('.')) == 3 else True)]
    
    print(f"After object frequency filtering (min freq: {min_object_freq}): {len(final_filtered)} relationships")
    
    # Convert synset format to human-readable form
    def get_readable_name(synset_name):
        try:
            if '.' in synset_name and len(synset_name.split('.')) == 3:
                synset = wn.synset(synset_name)
                return synset.lemma_names()[0]
            return synset_name
        except:
            return synset_name
    
    # Create a more readable version for humans
    readable_df = final_filtered.copy()
    readable_df['subject'] = readable_df['subject'].apply(get_readable_formatted_name)
    readable_df['object'] = readable_df['object'].apply(get_readable_formatted_name)
    
    # Format relation names too if needed
    readable_df['relation'] = readable_df['relation'].apply(format_term)
    
    readable_df = readable_df.drop(columns=['object_type'])
    
    return final_filtered, readable_df

def main(subject_threshold=100, object_threshold=100, relation_types=None):
    scene_graphs_path = "/Users/sammcmanagan/Desktop/Thesis/Model/data/vg/scene_graphs.json"
    output_path = f"wordnet_vg_relationships_s{subject_threshold}_o{object_threshold}.csv"
    readable_output_path = f"wordnet_vg_relationships_readable_s{subject_threshold}_o{object_threshold}.csv"
    
    # 1. Extract actual terms from Visual Genome and count them
    print("Loading scene graphs...")
    with open(scene_graphs_path, 'r') as f:
        scene_graphs = json.load(f)
    
    # Count actual terms used in Visual Genome
    term_counter = Counter()
    
    print(f"Processing {len(scene_graphs)} scene graphs to count terms...")
    for graph in tqdm(scene_graphs):
        # Process objects (these are the actual terms used in VG)
        if 'objects' in graph:
            for obj in graph['objects']:
                if 'names' in obj:
                    for name in obj['names']:
                        term_counter[name] += 1
    
    # Get frequent terms
    frequent_terms = {term for term, count in term_counter.items() 
                      if count >= min(subject_threshold, object_threshold)}
    
    print(f"Found {len(frequent_terms)} frequent terms")
    print("\nTop 20 terms by frequency:")
    for term, count in term_counter.most_common(20):
        print(f"  {term}: {count} occurrences")
    
    # 2. Generate WordNet relationships for these frequent terms
    wordnet_relationships = []
    
    for term in tqdm(frequent_terms):
        # Find all possible WordNet synsets for this term
        synsets = wn.synsets(term)
        
        for synset in synsets:
            # Only proceed if this is a noun synset
            if synset.pos() != 'n':
                continue
                
            # Extract valuable relationships
            # Hypernyms (is-a relationships)
            for hypernym in synset.hypernyms():
                for lemma in hypernym.lemma_names():
                    # Only keep if the hypernym is also frequent
                    if lemma in frequent_terms:
                        wordnet_relationships.append({
                            "subject": term,
                            "relation": "hypernym",
                            "object": lemma
                        })
            
            # Hyponyms (specific types)
            for hyponym in synset.hyponyms():
                for lemma in hyponym.lemma_names():
                    if lemma in frequent_terms:
                        wordnet_relationships.append({
                            "subject": term,
                            "relation": "hyponym",
                            "object": lemma
                        })
            
            # Synonyms (alternative names for the same concept)
            for other_lemma in synset.lemma_names():
                if other_lemma != term and other_lemma in frequent_terms:
                    wordnet_relationships.append({
                        "subject": term,
                        "relation": "synonym",
                        "object": other_lemma
        })
            
            # Meronyms (part-of relationships)
            for meronym in synset.part_meronyms():
                for lemma in meronym.lemma_names():
                    if lemma in frequent_terms:
                        wordnet_relationships.append({
                            "subject": term,
                            "relation": "part_meronym",
                            "object": lemma
                        })
            
            # Holonyms (whole-of relationships)
            for holonym in synset.part_holonyms():
                for lemma in holonym.lemma_names():
                    if lemma in frequent_terms:
                        wordnet_relationships.append({
                            "subject": term,
                            "relation": "part_holonym",
                            "object": lemma
                        })
    
    # Create dataframe and remove duplicates
    df = pd.DataFrame(wordnet_relationships)
    df = df.drop_duplicates()
    
    print(f"Generated {len(df)} WordNet relationships")
    
    # Create readable version with correct formatting
    readable_df = df.copy()
    readable_df['subject'] = readable_df['subject'].apply(format_term)
    readable_df['object'] = readable_df['object'].apply(format_term)
    readable_df['relation'] = readable_df['relation'].apply(format_term)
    
    # Save results
    df.to_csv(output_path, index=False)
    readable_df.to_csv(readable_output_path, index=False)
    
    # Display examples
    print("\nExample relationships:")
    print(readable_df.sample(min(10, len(readable_df))))

if __name__ == "__main__":
    # Example usage with different thresholds
    main(subject_threshold=30, object_threshold=30)
    
    # Uncomment to try different thresholds
    # main(subject_threshold=25, object_threshold=5)
    
    # Uncomment to filter for specific relation types
    # main(subject_threshold=50, object_threshold=10, relation_types=["hypernym", "part_meronym"])

In [9]:
df = pd.read_csv('/Users/sammcmanagan/Desktop/Thesis/Model/data/wordnet_vg_relationships_s50_o50.csv')

# Add this code to the end of your WordNet_Ontology.ipynb notebook

# Define SGG vocabulary
SGG_OBJECTS = [
    "__background__", "airplane", "animal", "arm", "bag", "banana", "basket", "beach", "bear", "bed", 
    "bench", "bike", "bird", "board", "boat", "book", "boot", "bottle", "bowl", "box", "boy", "branch", 
    "building", "bus", "cabinet", "cap", "car", "cat", "chair", "child", "clock", "coat", "counter", 
    "cow", "cup", "curtain", "desk", "dog", "door", "drawer", "ear", "elephant", "engine", "eye", 
    "face", "fence", "finger", "flag", "flower", "food", "fork", "fruit", "giraffe", "girl", "glass", 
    "glove", "guy", "hair", "hand", "handle", "hat", "head", "helmet", "hill", "horse", "house", 
    "jacket", "jean", "kid", "kite", "lady", "lamp", "laptop", "leaf", "leg", "letter", "light", 
    "logo", "man", "men", "motorcycle", "mountain", "mouth", "neck", "nose", "number", "orange", 
    "pant", "paper", "paw", "people", "person", "phone", "pillow", "pizza", "plane", "plant", 
    "plate", "player", "pole", "post", "pot", "racket", "railing", "rock", "roof", "room", 
    "screen", "seat", "sheep", "shelf", "shirt", "shoe", "short", "sidewalk", "sign", "sink", 
    "skateboard", "ski", "skier", "sneaker", "snow", "sock", "stand", "street", "surfboard", 
    "table", "tail", "tie", "tile", "tire", "toilet", "towel", "tower", "track", "train", 
    "tree", "truck", "trunk", "umbrella", "vase", "vegetable", "vehicle", "wave", "wheel", 
    "window", "windshield", "wing", "wire", "woman", "zebra"
]

sgg_vocab = [
    "airplane", "animal", "arm", "bag", "banana", "basket", "beach", "bear", "bed", 
    "bench", "bike", "bird", "board", "boat", "book", "boot", "bottle", "bowl", "box", "boy", "branch", 
    "building", "bus", "cabinet", "cap", "car", "cat", "chair", "child", "clock", "coat", "counter",    
    "cow", "cup", "curtain", "desk", "dog", "door", "drawer", "ear", "elephant", "engine", "eye",            
    "face", "fence", "finger", "flag", "flower", "food", "fork", "fruit", "giraffe", "girl", "glass",            
    "glove", "guy", "hair", "hand", "handle", "hat", "head", "helmet", "hill", "horse", "house",            
    "jacket", "jean", "kid", "kite", "lady", "lamp", "laptop", "leaf", "leg", "letter", "light",            
    "logo", "man", "men", "motorcycle", "mountain", "mouth", "neck", "nose", "number", "orange",            
    "pant", "paper", "paw", "people", "person", "phone", "pillow", "pizza", "plane", "plant",            
    "plate", "player", "pole", "post", "pot", "racket", "railing", "rock", "roof", "room",            
    "screen", "seat", "sheep", "shelf", "shirt", "shoe", "short", "sidewalk", "sign", "sink",            
    "skateboard", "ski", "skier", "sneaker", "snow", "sock", "stand", "street", "surfboard",            
    "table", "tail", "tie", "tile", "tire", "toilet", "towel", "tower", "track", "train",            
    "tree", "truck", "trunk", "umbrella", "vase", "vegetable", "vehicle", "wave", "wheel",            
    "window", "windshield", "wing", "wire", "woman", "zebra", "__background__", "above", 
    "across", "against", "along", "and", "at", "attached to", "behind", "belonging to", "between", 
    "carrying", "covered in", "covering", "eating", "flying in", "for", "from", "growing on", "hanging from", 
    "has", "holding", "in", "in front of", "laying on", "looking at", "lying on", "made of", "mounted on", "near", 
    "of", "on", "on back of", "over", "painted on", "parked on", "part of", "playing", "riding", "says", "sitting on", 
    "standing on", "to", "under", "using", "walking in", "walking on", "watching", "wearing", "wears", "with"]

# Remove __background__ as it's not a real object
SGG_OBJECTS = [obj for obj in SGG_OBJECTS if obj != "__background__"]

# Convert to set for faster lookup
SGG_VOCAB_SET = set(SGG_OBJECTS)

def filter_ontology_by_sgg_vocab(df, sgg_vocab_set, min_frequency=1):
    """
    Filter ontology relationships to only include those where both subject and object
    are in the SGG vocabulary.
    
    Args:
        df: DataFrame with ontology relationships
        sgg_vocab_set: Set of SGG vocabulary terms
        min_frequency: Minimum frequency threshold for keeping relationships
        
    Returns:
        Filtered DataFrame
    """
    # First, filter by SGG vocabulary
    sgg_filtered = df[
        df['subject'].isin(sgg_vocab_set) | 
        df['object'].isin(sgg_vocab_set)
    ]
    sgg_filtered = sgg_filtered[
        sgg_filtered['relation'].isin(['synonym', 'hypernym'])]
    
    # Count frequencies of relationships
    relationship_counts = sgg_filtered.groupby(['subject', 'relation', 'object']).size().reset_index(name='count')
    
    # Filter by minimum frequency if needed
    if min_frequency > 1:
        frequent_relationships = relationship_counts[relationship_counts['count'] >= min_frequency]
        # Merge back to get only frequent relationships
        sgg_filtered = sgg_filtered.merge(
            frequent_relationships[['subject', 'relation', 'object']], 
            on=['subject', 'relation', 'object'],
            how='inner'
        )
    
    # Remove duplicates
    sgg_filtered = sgg_filtered.drop_duplicates()
    
    print(f"Original ontology size: {len(df)}")
    print(f"After SGG vocabulary filtering: {len(sgg_filtered)}")
    print(f"Unique subjects: {sgg_filtered['subject'].nunique()}")
    print(f"Unique objects: {sgg_filtered['object'].nunique()}")
    print(f"Unique relations: {sgg_filtered['relation'].nunique()}")
    
    return sgg_filtered

# Apply SGG vocabulary filtering to your existing filtered ontology
sgg_filtered_df = filter_ontology_by_sgg_vocab(df, SGG_VOCAB_SET, min_frequency=1)

Original ontology size: 14966
After SGG vocabulary filtering: 1640
Unique subjects: 911
Unique objects: 442
Unique relations: 2


In [11]:
##Extracting vocabulary from cops-ref dataset
# Memory-efficient vocab extraction for large files
import re

sgg_vocab = [
    "airplane", "animal", "arm", "bag", "banana", "basket", "beach", "bear", "bed", 
    "bench", "bike", "bird", "board", "boat", "book", "boot", "bottle", "bowl", "box", "boy", "branch", 
    "building", "bus", "cabinet", "cap", "car", "cat", "chair", "child", "clock", "coat", "counter",    
    "cow", "cup", "curtain", "desk", "dog", "door", "drawer", "ear", "elephant", "engine", "eye",            
    "face", "fence", "finger", "flag", "flower", "food", "fork", "fruit", "giraffe", "girl", "glass",            
    "glove", "guy", "hair", "hand", "handle", "hat", "head", "helmet", "hill", "horse", "house",            
    "jacket", "jean", "kid", "kite", "lady", "lamp", "laptop", "leaf", "leg", "letter", "light",            
    "logo", "man", "men", "motorcycle", "mountain", "mouth", "neck", "nose", "number", "orange",            
    "pant", "paper", "paw", "people", "person", "phone", "pillow", "pizza", "plane", "plant",            
    "plate", "player", "pole", "post", "pot", "racket", "railing", "rock", "roof", "room",            
    "screen", "seat", "sheep", "shelf", "shirt", "shoe", "short", "sidewalk", "sign", "sink",            
    "skateboard", "ski", "skier", "sneaker", "snow", "sock", "stand", "street", "surfboard",            
    "table", "tail", "tie", "tile", "tire", "toilet", "towel", "tower", "track", "train",            
    "tree", "truck", "trunk", "umbrella", "vase", "vegetable", "vehicle", "wave", "wheel",            
    "window", "windshield", "wing", "wire", "woman", "zebra", "__background__", "above", 
    "across", "against", "along", "and", "at", "attached to", "behind", "belonging to", "between", 
    "carrying", "covered in", "covering", "eating", "flying in", "for", "from", "growing on", "hanging from", 
    "has", "holding", "in", "in front of", "laying on", "looking at", "lying on", "made of", "mounted on", "near", 
    "of", "on", "on back of", "over", "painted on", "parked on", "part of", "playing", "riding", "says", "sitting on", 
    "standing on", "to", "under", "using", "walking in", "walking on", "watching", "wearing", "wears", "with", 
    "white", "black", "blue", "green", "red", "brown", "yellow", "small", "large", 
    "wooden", "silver", "orange", "grey", "tall", "long", "dark", "pink", "standing", 
    "round", "tan", "glass", "here", "wood", "open", "purple", "short", "plastic", 
    "parked", "sitting", "walking", "striped", "brick", "young", "gold", "old", 
    "hanging", "empty", "on", "bright", "concrete", "cloudy", "colorful", "one", 
    "beige", "bare", "wet", "light", "square", "closed", "stone", "shiny", "thin", 
    "dirty", "flying", "smiling", "painted", "thick", "part", "sliced", "playing", 
    "tennis", "calm", "leather", "distant", "rectangular", "looking", "grassy", 
    "dry", "cement", "leafy", "wearing", "tiled", "man's", "baseball", "cooked", 
    "pictured", "curved", "decorative", "dead", "eating", "paper", "paved", "fluffy", 
    "lit", "back", "framed", "plaid", "dirt", "watching", "colored", "stuffed", 
    "clean", "in the picture", "steel", "stacked", "covered", "full", "three", 
    "street", "flat", "baby", "black and white", "beautiful", "ceramic", "present", 
    "grazing", "sandy", "golden", "blurry", "side", "chocolate", "wide", "growing", 
    "chrome", "cut", "bent", "train", "holding", "water", "up", "arched", "metallic", 
    "spotted", "folded", "electrical", "pointy", "running", "leafless", "electric", 
    "in background", "rusty", "furry", "traffic", "ripe", "behind", "laying", 
    "rocky", "tiny", "down", "fresh", "floral", "stainless steel", "high", "surfing", 
    "close", "off", "leaning", "moving", "multicolored", "woman's", "pair", "huge", 
    "some", "background", "chain link", "checkered", "top", "tree", "broken", 
    "maroon", "iron", "worn", "patterned", "ski", "overcast", "waiting", "rubber", 
    "riding", "skinny", "grass", "porcelain", "adult", "wire", "cloudless", "curly", 
    "cardboard", "jumping", "tile", "pointed", "blond", "cream", "four", "male", 
    "smooth", "hazy", "computer", "older", "pine", "raised", "many", "bald", 
    "snow covered", "skateboarding", "narrow", "reflective", "rear", "khaki", 
    "extended", "roman", "american"
]

def extract_vocab_large_file(file_path):
    vocab = set()
    
    # Process file in chunks to avoid memory issues
    with open(file_path, 'r') as f:
        chunk_size = 1024 * 1024  # 1MB chunks
        buffer = ""
        
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
                
            buffer += chunk
            
            # Process complete tokens arrays in buffer
            while '"tokens":' in buffer:
                start = buffer.find('"tokens":')
                if start == -1:
                    break
                    
                # Find the end of this tokens array
                bracket_start = buffer.find('[', start)
                if bracket_start == -1:
                    break
                    
                bracket_end = buffer.find(']', bracket_start)
                if bracket_end == -1:
                    break  # Incomplete array, keep in buffer
                
                # Extract this tokens array
                tokens_str = buffer[bracket_start:bracket_end+1]
                words = re.findall(r'"([^"]+)"', tokens_str)
                for word in words:
                    vocab.add(word.lower())
                
                # Remove processed part from buffer
                buffer = buffer[bracket_end+1:]
    
    return sorted(vocab)

# Alternative: Line-by-line processing (if file has line breaks)
def extract_vocab_by_lines(file_path):
    vocab = set()
    
    with open(file_path, 'r') as f:
        for line in f:
            if '"tokens":' in line:
                words = re.findall(r'"tokens":\s*\[([^\]]+)\]', line)
                for word_list in words:
                    tokens = re.findall(r'"([^"]+)"', word_list)
                    for token in tokens:
                        vocab.add(token.lower())
    
    return sorted(vocab)

# Usage
print("Processing large file...")
exp_vocab = extract_vocab_large_file('data.json')
print(f"Found {len(exp_vocab)} unique words")

Processing large file...
Found 1655 unique words


In [12]:
print(exp_vocab[:20])  # Display first 20 words

def quick_compare(my_vocab_list, sgg_objects):
    my_vocab_lower = [w.lower() for w in my_vocab_list]
    matches = [obj for obj in sgg_objects if obj.lower() in my_vocab_lower]
    non_matches = [obj for obj in sgg_objects if obj.lower() not in my_vocab_lower]
    print(f"Matches: {len(matches)}/{len(sgg_objects)} ({len(matches)/len(sgg_objects)*100:.1f}%)")
    print(f"Found: {', '.join(matches)}")
    print(f"Not found: {', '.join(non_matches)}")
    return matches

quick_compare(exp_vocab, SGG_OBJECTS)

['abandoned', 'about', 'above', 'abstract', 'abundant', 'across', 'adidas', 'adjusting', 'adult', 'against', 'air', 'aircraft', 'airplane', 'airport', 'alarm', 'alcohol', 'alert', 'along', 'alongside', 'aluminum']
Matches: 142/150 (94.7%)
Found: airplane, animal, arm, bag, banana, basket, beach, bear, bed, bench, bike, bird, board, boat, book, boot, bottle, bowl, box, boy, branch, building, bus, cabinet, cap, car, cat, chair, child, clock, coat, counter, cow, cup, curtain, desk, dog, door, drawer, ear, elephant, eye, face, fence, finger, flag, flower, food, fork, fruit, giraffe, girl, glass, glove, guy, hair, hand, hat, head, helmet, hill, horse, house, jacket, jean, kite, lady, lamp, laptop, leaf, leg, letter, light, logo, man, motorcycle, mountain, mouth, neck, nose, number, orange, pant, paper, paw, person, phone, pillow, pizza, plant, plate, player, pole, post, pot, racket, rock, roof, room, screen, seat, sheep, shelf, shirt, shoe, short, sidewalk, sign, sink, skateboard, ski, skie

['airplane',
 'animal',
 'arm',
 'bag',
 'banana',
 'basket',
 'beach',
 'bear',
 'bed',
 'bench',
 'bike',
 'bird',
 'board',
 'boat',
 'book',
 'boot',
 'bottle',
 'bowl',
 'box',
 'boy',
 'branch',
 'building',
 'bus',
 'cabinet',
 'cap',
 'car',
 'cat',
 'chair',
 'child',
 'clock',
 'coat',
 'counter',
 'cow',
 'cup',
 'curtain',
 'desk',
 'dog',
 'door',
 'drawer',
 'ear',
 'elephant',
 'eye',
 'face',
 'fence',
 'finger',
 'flag',
 'flower',
 'food',
 'fork',
 'fruit',
 'giraffe',
 'girl',
 'glass',
 'glove',
 'guy',
 'hair',
 'hand',
 'hat',
 'head',
 'helmet',
 'hill',
 'horse',
 'house',
 'jacket',
 'jean',
 'kite',
 'lady',
 'lamp',
 'laptop',
 'leaf',
 'leg',
 'letter',
 'light',
 'logo',
 'man',
 'motorcycle',
 'mountain',
 'mouth',
 'neck',
 'nose',
 'number',
 'orange',
 'pant',
 'paper',
 'paw',
 'person',
 'phone',
 'pillow',
 'pizza',
 'plant',
 'plate',
 'player',
 'pole',
 'post',
 'pot',
 'racket',
 'rock',
 'roof',
 'room',
 'screen',
 'seat',
 'sheep',
 'shelf',


In [5]:
def extract_nouns_nltk(vocab_list):
    """
    Extract nouns using NLTK part-of-speech tagging
    """

    import nltk
    nltk.download('averaged_perceptron_tagger_eng')
    from nltk import pos_tag, word_tokenize
    
    # Download required data (run once)
    # nltk.download('punkt')
    # nltk.download('averaged_perceptron_tagger')
    
    nouns = []
    
    for word in vocab_list:
        # Get POS tag for the word
        pos_tags = pos_tag([word])
        tag = pos_tags[0][1]
        
        # Check if it's a noun (NN, NNS, NNP, NNPS)
        if tag.startswith('NN'):
            nouns.append(word)
            
    return nouns
        
nouns = extract_nouns_nltk(exp_vocab)
print("Extracted nouns:", len(nouns))

Extracted nouns: 1215


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/sammcmanagan/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [22]:
def check_nouns_in_sgg(vocab_nouns, sgg_df):
    """
    Check how many vocabulary nouns appear in SGG subject/object columns
    """
    
    # Get all unique entities from subject and object columns
    all_entities = set()
    all_entities.update(sgg_df['subject'].str.lower())
    all_entities.update(sgg_df['object'].str.lower())
    
    # Check each noun
    found = []
    not_found = []
    
    for noun in vocab_nouns:
        if noun.lower() in all_entities:
            found.append(noun)
        else:
            not_found.append(noun)
    
    # Print results
    print(f"Nouns checked: {len(vocab_nouns)}")
    print(f"Found in SGG: {len(found)}")
    print(f"Coverage: {len(found)/len(vocab_nouns)*100:.1f}%")
    print(f"Found: {', '.join(found)}")
    print(f"Not found: {', '.join(not_found)}")
    
    return found

found_nouns = check_nouns_in_sgg(nouns, complete_ontology)
print("Found nouns in SGG:", len(found_nouns))

Nouns checked: 1215
Found in SGG: 1202
Coverage: 98.9%
Found: abstract, abundant, adult, air, aircraft, airplane, airport, alarm, alcohol, alert, aluminum, ambulance, analog, animal, antelope, antenna, antique, apartment, apple, appliance, apron, aquarium, arm, armchair, arrow, artichoke, artwork, asphalt, athlete, atop, audience, avocado, baby, backpack, bacon, bag, bagel, bakery, baking, balcony, bald, balding, ball, balloon, bamboo, banana, bandage, bandana, bar, barbed, bare, barefoot, barn, barren, barrier, baseball, basil, basket, bat, bath, bathroom, bathtub, batter, batting, beach, bead, beak, bean, bear, beard, beautiful, bed, bedding, bedroom, bedspread, beef, beer, beet, beige, bell, belt, bench, bending, beneath, bent, berry, beside, beverage, bicycle, bike, biker, bikini, binder, bird, birthday, biscuit, bison, biting, blank, blanket, blazer, bleacher, blender, blind, block, blond, blood, blooming, blossom, blouse, blowing, blue, blueberry, blurry, board, boarding, boat, b

In [24]:
complete_ontology.to_csv('final_ontology.csv', index=False)

In [20]:
# Final cleanup - relationships for remaining valid terms
# Ignoring: adidas, nike, coke, ipod, wii (brands)
# Ignoring: asparagu, bu, cros, dres, ga, octopu, tenni, slouse (misspellings)

final_cleanup_ontology = [
    # ========== MATERIALS ==========
    {'subject': 'leather', 'relation': 'hypernym', 'object': 'material'},
    {'subject': 'plastic', 'relation': 'hypernym', 'object': 'material'},
    {'subject': 'porcelain', 'relation': 'hypernym', 'object': 'ceramic'},
    
    # ========== FOOD ==========
    {'subject': 'peanut', 'relation': 'hypernym', 'object': 'nut'},
    {'subject': 'pickle', 'relation': 'hypernym', 'object': 'food'},
    {'subject': 'crumb', 'relation': 'hypernym', 'object': 'food'},
    {'subject': 'crust', 'relation': 'hypernym', 'object': 'food'},
    {'subject': 'champagne', 'relation': 'hypernym', 'object': 'wine'},
    
    # ========== PEOPLE ==========
    {'subject': 'biker', 'relation': 'hypernym', 'object': 'person'},
    {'subject': 'snowboarder', 'relation': 'hypernym', 'object': 'person'},
    
    # ========== ANIMALS ==========
    {'subject': 'dinosaur', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'tabby', 'relation': 'hypernym', 'object': 'cat'},
    
    # ========== OBJECTS ==========
    {'subject': 'pencil', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'piano', 'relation': 'hypernym', 'object': 'instrument'},
    {'subject': 'garland', 'relation': 'hypernym', 'object': 'decoration'},
    {'subject': 'skate', 'relation': 'hypernym', 'object': 'shoe'},
    
    # ========== BODY/NATURE ==========
    {'subject': 'blood', 'relation': 'hypernym', 'object': 'liquid'},
    {'subject': 'palm', 'relation': 'hypernym', 'object': 'hand'},
    {'subject': 'palm', 'relation': 'hypernym', 'object': 'tree'},
    
    # ========== DESCRIPTIVE ==========
    {'subject': 'barbed', 'relation': 'synonym', 'object': 'sharp'},
    {'subject': 'chain-link', 'relation': 'hypernym', 'object': 'fence'},
    {'subject': 'incomplete', 'relation': 'synonym', 'object': 'unfinished'},
    {'subject': 'roman', 'relation': 'synonym', 'object': 'classical'},
    
    # ========== ABSTRACT/CONCEPTS ==========
    {'subject': 'picnic', 'relation': 'hypernym', 'object': 'meal'},
    {'subject': 'polo', 'relation': 'hypernym', 'object': 'sport'},
    {'subject': 'fifth', 'relation': 'hypernym', 'object': 'number'},
    {'subject': 'half', 'relation': 'hypernym', 'object': 'part'},
    {'subject': 'hour', 'relation': 'hypernym', 'object': 'time'},
    
    # ========== ACTIONS/DIRECTIONS ==========
    {'subject': 'flop', 'relation': 'synonym', 'object': 'fall'},
    {'subject': 'forth', 'relation': 'synonym', 'object': 'forward'},
    {'subject': 'left', 'relation': 'synonym', 'object': 'direction'},
    {'subject': 'towards', 'relation': 'synonym', 'object': 'direction'},
    
    # ========== COMPOUND TERMS ==========
    {'subject': 'decker', 'relation': 'hypernym', 'object': 'level'},  # as in double-decker
]

# Add these final relationships
final_cleanup_df = pd.DataFrame(final_cleanup_ontology)
complete_ontology = pd.concat([final_ontology, final_cleanup_df], ignore_index=True)
complete_ontology = complete_ontology.drop_duplicates()

print(f"Added {len(final_cleanup_ontology)} final relationships")
print(f"Complete ontology: {len(complete_ontology)} total relationships")
print(f"Ignored brand names: adidas, nike, coke, ipod, wii")
print(f"Ignored misspellings: asparagu, bu, cros, dres, ga, octopu, tenni, slouse")

Added 33 final relationships
Complete ontology: 1363 total relationships
Ignored brand names: adidas, nike, coke, ipod, wii
Ignored misspellings: asparagu, bu, cros, dres, ga, octopu, tenni, slouse


In [18]:
# Additional ontology relationships for remaining "not found" terms
# Ignoring obvious misspellings: asparagu, bu, cros, dres, ga, octopu, tenni, slouse
# Ignoring brand names: adidas, nike, coke, ipod, wii

additional_ontology = [
    # ========== OBJECTS & TOOLS ==========
    {'subject': 'arrow', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'cage', 'relation': 'hypernym', 'object': 'container'},
    {'subject': 'bomb', 'relation': 'hypernym', 'object': 'weapon'},
    {'subject': 'bouquet', 'relation': 'hypernym', 'object': 'flower'},
    {'subject': 'briefcase', 'relation': 'hypernym', 'object': 'bag'},
    {'subject': 'buoy', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'burner', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'cactus', 'relation': 'hypernym', 'object': 'plant'},
    {'subject': 'chandelier', 'relation': 'hypernym', 'object': 'lamp'},
    {'subject': 'chimney', 'relation': 'hypernym', 'object': 'pipe'},
    {'subject': 'cone', 'relation': 'hypernym', 'object': 'shape'},
    {'subject': 'cooker', 'relation': 'hypernym', 'object': 'appliance'},
    {'subject': 'countertop', 'relation': 'hypernym', 'object': 'surface'},
    {'subject': 'crane', 'relation': 'hypernym', 'object': 'machine'},
    {'subject': 'crane', 'relation': 'hypernym', 'object': 'bird'},
    {'subject': 'diamond', 'relation': 'hypernym', 'object': 'stone'},
    {'subject': 'dome', 'relation': 'hypernym', 'object': 'roof'},
    {'subject': 'donkey', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'drain', 'relation': 'hypernym', 'object': 'pipe'},
    {'subject': 'drainer', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'drum', 'relation': 'hypernym', 'object': 'instrument'},
    {'subject': 'dvd', 'relation': 'hypernym', 'object': 'disc'},
    {'subject': 'fireplace', 'relation': 'hypernym', 'object': 'fire'},
    {'subject': 'fountain', 'relation': 'hypernym', 'object': 'water'},
    {'subject': 'guitar', 'relation': 'hypernym', 'object': 'instrument'},
    {'subject': 'horn', 'relation': 'hypernym', 'object': 'instrument'},
    {'subject': 'lock', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'manhole', 'relation': 'hypernym', 'object': 'hole'},
    {'subject': 'mannequin', 'relation': 'hypernym', 'object': 'figure'},
    {'subject': 'marker', 'relation': 'hypernym', 'object': 'pen'},
    {'subject': 'meter', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'net', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'parachute', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'pillar', 'relation': 'hypernym', 'object': 'support'},
    {'subject': 'pin', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'pipe', 'relation': 'hypernym', 'object': 'tube'},
    {'subject': 'planter', 'relation': 'hypernym', 'object': 'container'},
    {'subject': 'propeller', 'relation': 'hypernym', 'object': 'blade'},
    {'subject': 'pump', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'racket', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'radiator', 'relation': 'hypernym', 'object': 'heater'},
    {'subject': 'satellite', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'shield', 'relation': 'hypernym', 'object': 'protection'},
    {'subject': 'silverware', 'relation': 'hypernym', 'object': 'utensil'},
    {'subject': 'sink', 'relation': 'hypernym', 'object': 'basin'},
    {'subject': 'sponge', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'staircase', 'relation': 'hypernym', 'object': 'stair'},
    {'subject': 'sticker', 'relation': 'hypernym', 'object': 'label'},
    {'subject': 'straw', 'relation': 'hypernym', 'object': 'tube'},
    {'subject': 'trunk', 'relation': 'hypernym', 'object': 'box'},
    {'subject': 'turbine', 'relation': 'hypernym', 'object': 'machine'},
    
    # ========== PEOPLE & ROLES ==========
    {'subject': 'batter', 'relation': 'hypernym', 'object': 'player'},
    {'subject': 'catcher', 'relation': 'hypernym', 'object': 'player'},
    {'subject': 'chef', 'relation': 'hypernym', 'object': 'worker'},
    {'subject': 'cowboy', 'relation': 'hypernym', 'object': 'person'},
    {'subject': 'cyclist', 'relation': 'hypernym', 'object': 'person'},
    {'subject': 'driver', 'relation': 'hypernym', 'object': 'person'},
    {'subject': 'father', 'relation': 'hypernym', 'object': 'man'},
    {'subject': 'jockey', 'relation': 'hypernym', 'object': 'person'},
    {'subject': 'mother', 'relation': 'hypernym', 'object': 'woman'},
    {'subject': 'passenger', 'relation': 'hypernym', 'object': 'person'},
    {'subject': 'photographer', 'relation': 'hypernym', 'object': 'person'},
    {'subject': 'pilot', 'relation': 'hypernym', 'object': 'person'},
    {'subject': 'spectator', 'relation': 'hypernym', 'object': 'person'},
    {'subject': 'student', 'relation': 'hypernym', 'object': 'person'},
    {'subject': 'surfer', 'relation': 'hypernym', 'object': 'person'},
    {'subject': 'umpire', 'relation': 'hypernym', 'object': 'person'},
    {'subject': 'waiter', 'relation': 'hypernym', 'object': 'person'},
    
    # ========== MATERIALS & SUBSTANCES ==========
    {'subject': 'foil', 'relation': 'hypernym', 'object': 'metal'},
    {'subject': 'flour', 'relation': 'hypernym', 'object': 'food'},
    {'subject': 'fur', 'relation': 'hypernym', 'object': 'hair'},
    {'subject': 'hay', 'relation': 'hypernym', 'object': 'grass'},
    {'subject': 'medicine', 'relation': 'hypernym', 'object': 'drug'},
    {'subject': 'seaweed', 'relation': 'hypernym', 'object': 'plant'},
    {'subject': 'shampoo', 'relation': 'hypernym', 'object': 'soap'},
    {'subject': 'silver', 'relation': 'hypernym', 'object': 'metal'},
    {'subject': 'snow', 'relation': 'hypernym', 'object': 'ice'},
    {'subject': 'soap', 'relation': 'hypernym', 'object': 'cleaner'},
    {'subject': 'toothpaste', 'relation': 'hypernym', 'object': 'cleaner'},
    
    # ========== NATURAL ELEMENTS ==========
    {'subject': 'cliff', 'relation': 'hypernym', 'object': 'rock'},
    {'subject': 'fog', 'relation': 'hypernym', 'object': 'cloud'},
    {'subject': 'leaf', 'relation': 'hypernym', 'object': 'plant'},
    {'subject': 'moon', 'relation': 'hypernym', 'object': 'celestial'},
    {'subject': 'pine', 'relation': 'hypernym', 'object': 'tree'},
    {'subject': 'puddle', 'relation': 'hypernym', 'object': 'water'},
    {'subject': 'rain', 'relation': 'hypernym', 'object': 'water'},
    {'subject': 'rainbow', 'relation': 'hypernym', 'object': 'arc'},
    {'subject': 'star', 'relation': 'hypernym', 'object': 'celestial'},
    {'subject': 'sun', 'relation': 'hypernym', 'object': 'star'},
    {'subject': 'tree', 'relation': 'hypernym', 'object': 'plant'},
    {'subject': 'weed', 'relation': 'hypernym', 'object': 'plant'},
    {'subject': 'wildflower', 'relation': 'hypernym', 'object': 'flower'},
    
    # ========== PLACES & LOCATIONS ==========
    {'subject': 'capital', 'relation': 'hypernym', 'object': 'city'},
    {'subject': 'cockpit', 'relation': 'hypernym', 'object': 'room'},
    {'subject': 'stage', 'relation': 'hypernym', 'object': 'platform'},
    {'subject': 'street', 'relation': 'hypernym', 'object': 'road'},
    
    # ========== DESCRIPTIVE ATTRIBUTES (SYNONYMS) ==========
    {'subject': 'abundant', 'relation': 'synonym', 'object': 'many'},
    {'subject': 'alert', 'relation': 'synonym', 'object': 'awake'},
    {'subject': 'analog', 'relation': 'synonym', 'object': 'traditional'},
    {'subject': 'antique', 'relation': 'synonym', 'object': 'old'},
    {'subject': 'bare', 'relation': 'synonym', 'object': 'empty'},
    {'subject': 'barefoot', 'relation': 'synonym', 'object': 'bare'},
    {'subject': 'blooming', 'relation': 'synonym', 'object': 'flowering'},
    {'subject': 'blowing', 'relation': 'synonym', 'object': 'windy'},
    {'subject': 'clumped', 'relation': 'synonym', 'object': 'grouped'},
    {'subject': 'coarse', 'relation': 'synonym', 'object': 'rough'},
    {'subject': 'cobblestone', 'relation': 'synonym', 'object': 'stone'},
    {'subject': 'cordless', 'relation': 'synonym', 'object': 'wireless'},
    {'subject': 'creamy', 'relation': 'synonym', 'object': 'smooth'},
    {'subject': 'crooked', 'relation': 'synonym', 'object': 'bent'},
    {'subject': 'crusty', 'relation': 'synonym', 'object': 'hard'},
    {'subject': 'curvy', 'relation': 'synonym', 'object': 'curved'},
    {'subject': 'digital', 'relation': 'synonym', 'object': 'electronic'},
    {'subject': 'directional', 'relation': 'synonym', 'object': 'pointing'},
    {'subject': 'dull', 'relation': 'synonym', 'object': 'boring'},
    {'subject': 'dusty', 'relation': 'synonym', 'object': 'dirty'},
    {'subject': 'evergreen', 'relation': 'synonym', 'object': 'green'},
    {'subject': 'fake', 'relation': 'synonym', 'object': 'artificial'},
    {'subject': 'fine', 'relation': 'synonym', 'object': 'good'},
    {'subject': 'fluorescent', 'relation': 'synonym', 'object': 'bright'},
    {'subject': 'foamy', 'relation': 'synonym', 'object': 'bubbly'},
    {'subject': 'frozen', 'relation': 'synonym', 'object': 'cold'},
    {'subject': 'gloomy', 'relation': 'synonym', 'object': 'dark'},
    {'subject': 'grassy', 'relation': 'synonym', 'object': 'green'},
    {'subject': 'greasy', 'relation': 'synonym', 'object': 'oily'},
    {'subject': 'grouped', 'relation': 'synonym', 'object': 'together'},
    {'subject': 'handmade', 'relation': 'synonym', 'object': 'crafted'},
    {'subject': 'hidden', 'relation': 'synonym', 'object': 'covered'},
    {'subject': 'hollow', 'relation': 'synonym', 'object': 'empty'},
    {'subject': 'homemade', 'relation': 'synonym', 'object': 'crafted'},
    {'subject': 'horizontal', 'relation': 'synonym', 'object': 'flat'},
    {'subject': 'intricate', 'relation': 'synonym', 'object': 'complex'},
    {'subject': 'irregular', 'relation': 'synonym', 'object': 'uneven'},
    {'subject': 'juicy', 'relation': 'synonym', 'object': 'wet'},
    {'subject': 'leafless', 'relation': 'synonym', 'object': 'bare'},
    {'subject': 'leafy', 'relation': 'synonym', 'object': 'green'},
    {'subject': 'lit', 'relation': 'synonym', 'object': 'bright'},
    {'subject': 'lush', 'relation': 'synonym', 'object': 'thick'},
    {'subject': 'messy', 'relation': 'synonym', 'object': 'dirty'},
    {'subject': 'neat', 'relation': 'synonym', 'object': 'clean'},
    {'subject': 'ornate', 'relation': 'synonym', 'object': 'decorative'},
    {'subject': 'overgrown', 'relation': 'synonym', 'object': 'wild'},
    {'subject': 'packed', 'relation': 'synonym', 'object': 'full'},
    {'subject': 'pale', 'relation': 'synonym', 'object': 'light'},
    {'subject': 'patchy', 'relation': 'synonym', 'object': 'uneven'},
    {'subject': 'pointy', 'relation': 'synonym', 'object': 'sharp'},
    {'subject': 'polar', 'relation': 'synonym', 'object': 'cold'},
    {'subject': 'protective', 'relation': 'synonym', 'object': 'safe'},
    {'subject': 'raw', 'relation': 'synonym', 'object': 'uncooked'},
    {'subject': 'rectangular', 'relation': 'synonym', 'object': 'square'},
    {'subject': 'reflective', 'relation': 'synonym', 'object': 'shiny'},
    {'subject': 'rocky', 'relation': 'synonym', 'object': 'stone'},
    {'subject': 'rusty', 'relation': 'synonym', 'object': 'old'},
    {'subject': 'sad', 'relation': 'synonym', 'object': 'unhappy'},
    {'subject': 'sandy', 'relation': 'synonym', 'object': 'sand'},
    {'subject': 'sewn', 'relation': 'synonym', 'object': 'stitched'},
    {'subject': 'shirtless', 'relation': 'synonym', 'object': 'bare'},
    {'subject': 'sleeveless', 'relation': 'synonym', 'object': 'bare'},
    {'subject': 'sour', 'relation': 'synonym', 'object': 'bitter'},
    {'subject': 'spiky', 'relation': 'synonym', 'object': 'sharp'},
    {'subject': 'spiral', 'relation': 'synonym', 'object': 'curved'},
    {'subject': 'square', 'relation': 'hypernym', 'object': 'shape'},
    {'subject': 'sticky', 'relation': 'synonym', 'object': 'adhesive'},
    {'subject': 'stuffed', 'relation': 'synonym', 'object': 'full'},
    {'subject': 'sturdy', 'relation': 'synonym', 'object': 'strong'},
    {'subject': 'tasty', 'relation': 'synonym', 'object': 'good'},
    {'subject': 'tight', 'relation': 'synonym', 'object': 'fitted'},
    {'subject': 'triangular', 'relation': 'hypernym', 'object': 'shape'},
    {'subject': 'unlit', 'relation': 'synonym', 'object': 'dark'},
    {'subject': 'vacant', 'relation': 'synonym', 'object': 'empty'},
    {'subject': 'warm', 'relation': 'synonym', 'object': 'hot'},
    {'subject': 'wavy', 'relation': 'synonym', 'object': 'curved'},
    {'subject': 'wispy', 'relation': 'synonym', 'object': 'thin'},
    {'subject': 'worn', 'relation': 'synonym', 'object': 'old'},
    
    # ========== ACTIONS & STATES ==========
    {'subject': 'biting', 'relation': 'synonym', 'object': 'eating'},
    {'subject': 'batting', 'relation': 'synonym', 'object': 'hitting'},
    {'subject': 'contain', 'relation': 'synonym', 'object': 'hold'},
    {'subject': 'drawn', 'relation': 'synonym', 'object': 'pulled'},
    {'subject': 'kept', 'relation': 'synonym', 'object': 'stored'},
    
    # ========== ABSTRACT CONCEPTS ==========
    {'subject': 'audience', 'relation': 'hypernym', 'object': 'people'},
    {'subject': 'birthday', 'relation': 'hypernym', 'object': 'day'},
    {'subject': 'cell', 'relation': 'hypernym', 'object': 'room'},
    {'subject': 'christmas', 'relation': 'hypernym', 'object': 'holiday'},
    {'subject': 'entertainment', 'relation': 'synonym', 'object': 'fun'},
    {'subject': 'game', 'relation': 'hypernym', 'object': 'toy'},
    {'subject': 'gender', 'relation': 'hypernym', 'object': 'type'},
    {'subject': 'license', 'relation': 'hypernym', 'object': 'permit'},
    {'subject': 'logo', 'relation': 'hypernym', 'object': 'symbol'},
    {'subject': 'name', 'relation': 'hypernym', 'object': 'word'},
    {'subject': 'note', 'relation': 'hypernym', 'object': 'paper'},
    {'subject': 'stop', 'relation': 'synonym', 'object': 'end'},
    {'subject': 'team', 'relation': 'hypernym', 'object': 'group'},
    {'subject': 'traffic', 'relation': 'hypernym', 'object': 'cars'},
    {'subject': 'wedding', 'relation': 'hypernym', 'object': 'ceremony'},
    {'subject': 'winter', 'relation': 'hypernym', 'object': 'season'},
    {'subject': 'word', 'relation': 'hypernym', 'object': 'text'},
    
    # ========== PARTS & COMPONENTS ==========
    {'subject': 'binder', 'relation': 'hypernym', 'object': 'container'},
    {'subject': 'conditioner', 'relation': 'hypernym', 'object': 'product'},
    {'subject': 'feeder', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'gate', 'relation': 'hypernym', 'object': 'door'},
    {'subject': 'headband', 'relation': 'hypernym', 'object': 'band'},
    {'subject': 'holder', 'relation': 'hypernym', 'object': 'container'},
    {'subject': 'lift', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'lipstick', 'relation': 'hypernym', 'object': 'makeup'},
    {'subject': 'maker', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'opening', 'relation': 'hypernym', 'object': 'hole'},
    {'subject': 'pole', 'relation': 'hypernym', 'object': 'stick'},
    {'subject': 'register', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'stump', 'relation': 'hypernym', 'object': 'tree'},
]

# Add to your existing ontology
new_additional_df = pd.DataFrame(additional_ontology)
final_ontology = pd.concat([expanded_ontology, new_additional_df], ignore_index=True)
final_ontology = final_ontology.drop_duplicates()

print(f"Added {len(additional_ontology)} more relationships")
print(f"Final ontology: {len(final_ontology)} total relationships")

Added 209 more relationships
Final ontology: 1330 total relationships


In [16]:
# Convert the new relationships to a DataFrame
new_ontology_df = pd.DataFrame(not_found_ontology)

# Add to your existing ontology
expanded_ontology = pd.concat([simple_ontology, new_ontology_df], ignore_index=True)

# Remove any duplicates (just in case)
expanded_ontology = expanded_ontology.drop_duplicates()

In [15]:
# Complete ontology for "not found" terms
# Only includes relationships where both terms are in your vocabulary lists

not_found_ontology = [
    # ========== ANIMALS ==========
    # Large mammals
    {'subject': 'bear', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'elephant', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'horse', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'cow', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'deer', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'sheep', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'goat', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'bull', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'calf', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'lamb', 'relation': 'hypernym', 'object': 'animal'},
    
    # Farm animals
    {'subject': 'cow', 'relation': 'synonym', 'object': 'bull'},
    {'subject': 'sheep', 'relation': 'synonym', 'object': 'lamb'},
    
    # Wild animals
    {'subject': 'antelope', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'bison', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'giraffe', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'rhino', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'zebra', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'panda', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'monkey', 'relation': 'hypernym', 'object': 'animal'},
    
    # Small animals
    {'subject': 'bunny', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'squirrel', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'frog', 'relation': 'hypernym', 'object': 'animal'},
    {'subject': 'crab', 'relation': 'hypernym', 'object': 'animal'},
    
    # Birds
    {'subject': 'duck', 'relation': 'hypernym', 'object': 'bird'},
    {'subject': 'goose', 'relation': 'hypernym', 'object': 'bird'},
    {'subject': 'pigeon', 'relation': 'hypernym', 'object': 'bird'},
    {'subject': 'seagull', 'relation': 'hypernym', 'object': 'bird'},
    {'subject': 'flamingo', 'relation': 'hypernym', 'object': 'bird'},
    {'subject': 'swan', 'relation': 'hypernym', 'object': 'bird'},
    {'subject': 'turkey', 'relation': 'hypernym', 'object': 'bird'},
    
    # Insects
    {'subject': 'butterfly', 'relation': 'hypernym', 'object': 'animal'},
    
    # ========== FOOD ==========
    # Fruits
    {'subject': 'apple', 'relation': 'hypernym', 'object': 'fruit'},
    {'subject': 'avocado', 'relation': 'hypernym', 'object': 'fruit'},
    {'subject': 'grape', 'relation': 'hypernym', 'object': 'fruit'},
    {'subject': 'grapefruit', 'relation': 'hypernym', 'object': 'fruit'},
    {'subject': 'lemon', 'relation': 'hypernym', 'object': 'fruit'},
    {'subject': 'lime', 'relation': 'hypernym', 'object': 'fruit'},
    {'subject': 'mango', 'relation': 'hypernym', 'object': 'fruit'},
    {'subject': 'peach', 'relation': 'hypernym', 'object': 'fruit'},
    {'subject': 'pear', 'relation': 'hypernym', 'object': 'fruit'},
    {'subject': 'papaya', 'relation': 'hypernym', 'object': 'fruit'},
    {'subject': 'pomegranate', 'relation': 'hypernym', 'object': 'fruit'},
    {'subject': 'tangerine', 'relation': 'hypernym', 'object': 'fruit'},
    {'subject': 'raisin', 'relation': 'hypernym', 'object': 'fruit'},
    {'subject': 'raspberry', 'relation': 'hypernym', 'object': 'fruit'},
    
    # Vegetables
    {'subject': 'broccoli', 'relation': 'hypernym', 'object': 'vegetable'},
    {'subject': 'carrot', 'relation': 'hypernym', 'object': 'vegetable'},
    {'subject': 'cauliflower', 'relation': 'hypernym', 'object': 'vegetable'},
    {'subject': 'eggplant', 'relation': 'hypernym', 'object': 'vegetable'},
    {'subject': 'garlic', 'relation': 'hypernym', 'object': 'vegetable'},
    {'subject': 'mushroom', 'relation': 'hypernym', 'object': 'vegetable'},
    {'subject': 'potato', 'relation': 'hypernym', 'object': 'vegetable'},
    {'subject': 'tomato', 'relation': 'hypernym', 'object': 'vegetable'},
    {'subject': 'zucchini', 'relation': 'hypernym', 'object': 'vegetable'},
    {'subject': 'pea', 'relation': 'hypernym', 'object': 'vegetable'},
    {'subject': 'bean', 'relation': 'hypernym', 'object': 'vegetable'},
    {'subject': 'pepper', 'relation': 'hypernym', 'object': 'vegetable'},
    
    # Meat/Protein
    {'subject': 'bacon', 'relation': 'hypernym', 'object': 'meat'},
    {'subject': 'beef', 'relation': 'hypernym', 'object': 'meat'},
    {'subject': 'chicken', 'relation': 'hypernym', 'object': 'meat'},
    {'subject': 'ham', 'relation': 'hypernym', 'object': 'meat'},
    {'subject': 'shrimp', 'relation': 'hypernym', 'object': 'meat'},
    {'subject': 'fish', 'relation': 'hypernym', 'object': 'meat'},
    {'subject': 'egg', 'relation': 'hypernym', 'object': 'food'},
    {'subject': 'tofu', 'relation': 'hypernym', 'object': 'food'},
    
    # Prepared foods
    {'subject': 'hotdog', 'relation': 'hypernym', 'object': 'food'},
    {'subject': 'stew', 'relation': 'hypernym', 'object': 'food'},
    {'subject': 'soup', 'relation': 'hypernym', 'object': 'dish'},
    {'subject': 'macaroni', 'relation': 'hypernym', 'object': 'food'},
    {'subject': 'brownie', 'relation': 'hypernym', 'object': 'dessert'},
    {'subject': 'donut', 'relation': 'hypernym', 'object': 'dessert'},
    {'subject': 'muffin', 'relation': 'hypernym', 'object': 'dessert'},
    
    # Dairy
    {'subject': 'cream', 'relation': 'hypernym', 'object': 'food'},
    {'subject': 'feta', 'relation': 'hypernym', 'object': 'cheese'},
    
    # Condiments/liquids
    {'subject': 'ketchup', 'relation': 'hypernym', 'object': 'sauce'},
    {'subject': 'mustard', 'relation': 'hypernym', 'object': 'sauce'},
    {'subject': 'syrup', 'relation': 'hypernym', 'object': 'food'},
    {'subject': 'honey', 'relation': 'hypernym', 'object': 'food'},
    {'subject': 'oil', 'relation': 'hypernym', 'object': 'liquid'},
    {'subject': 'juice', 'relation': 'hypernym', 'object': 'drink'},
    {'subject': 'lemonade', 'relation': 'hypernym', 'object': 'drink'},
    {'subject': 'beer', 'relation': 'hypernym', 'object': 'alcohol'},
    
    # Spices/seasonings
    {'subject': 'cinnamon', 'relation': 'hypernym', 'object': 'spice'},
    {'subject': 'vanilla', 'relation': 'hypernym', 'object': 'spice'},
    
    # ========== COLORS ==========
    {'subject': 'blue', 'relation': 'hypernym', 'object': 'color'},
    {'subject': 'orange', 'relation': 'hypernym', 'object': 'color'},
    {'subject': 'pink', 'relation': 'hypernym', 'object': 'color'},
    {'subject': 'purple', 'relation': 'hypernym', 'object': 'color'},
    {'subject': 'yellow', 'relation': 'hypernym', 'object': 'color'},
    {'subject': 'beige', 'relation': 'hypernym', 'object': 'color'},
    {'subject': 'cream', 'relation': 'hypernym', 'object': 'color'},
    {'subject': 'maroon', 'relation': 'hypernym', 'object': 'color'},
    {'subject': 'navy', 'relation': 'hypernym', 'object': 'color'},
    {'subject': 'tan', 'relation': 'hypernym', 'object': 'color'},
    {'subject': 'bronze', 'relation': 'hypernym', 'object': 'color'},
    {'subject': 'ivory', 'relation': 'hypernym', 'object': 'color'},
    {'subject': 'neon', 'relation': 'hypernym', 'object': 'color'},
    {'subject': 'pastel', 'relation': 'hypernym', 'object': 'color'},
    {'subject': 'vibrant', 'relation': 'hypernym', 'object': 'color'},
    
    # ========== BODY PARTS ==========
    {'subject': 'beard', 'relation': 'hypernym', 'object': 'hair'},
    {'subject': 'mustache', 'relation': 'hypernym', 'object': 'hair'},
    {'subject': 'ear', 'relation': 'hypernym', 'object': 'body'},
    {'subject': 'elbow', 'relation': 'hypernym', 'object': 'arm'},
    {'subject': 'knee', 'relation': 'hypernym', 'object': 'leg'},
    {'subject': 'neck', 'relation': 'hypernym', 'object': 'body'},
    {'subject': 'waist', 'relation': 'hypernym', 'object': 'body'},
    {'subject': 'wrist', 'relation': 'hypernym', 'object': 'arm'},
    {'subject': 'lip', 'relation': 'hypernym', 'object': 'face'},
    {'subject': 'tongue', 'relation': 'hypernym', 'object': 'mouth'},
    
    # ========== CLOTHING & ACCESSORIES ==========
    {'subject': 'boot', 'relation': 'hypernym', 'object': 'shoe'},
    {'subject': 'cap', 'relation': 'hypernym', 'object': 'hat'},
    {'subject': 'helmet', 'relation': 'hypernym', 'object': 'hat'},
    {'subject': 'bikini', 'relation': 'hypernym', 'object': 'swimsuit'},
    {'subject': 'jersey', 'relation': 'hypernym', 'object': 'shirt'},
    {'subject': 't-shirt', 'relation': 'hypernym', 'object': 'shirt'},
    {'subject': 'sweatshirt', 'relation': 'hypernym', 'object': 'shirt'},
    {'subject': 'uniform', 'relation': 'hypernym', 'object': 'clothing'},
    {'subject': 'apron', 'relation': 'hypernym', 'object': 'clothing'},
    {'subject': 'bandana', 'relation': 'hypernym', 'object': 'clothing'},
    {'subject': 'belt', 'relation': 'hypernym', 'object': 'clothing'},
    {'subject': 'necklace', 'relation': 'hypernym', 'object': 'jewelry'},
    {'subject': 'goggle', 'relation': 'hypernym', 'object': 'eyewear'},
    {'subject': 'sunglass', 'relation': 'hypernym', 'object': 'eyewear'},
    {'subject': 'wetsuit', 'relation': 'hypernym', 'object': 'clothing'},
    {'subject': 'snowsuit', 'relation': 'hypernym', 'object': 'clothing'},
    {'subject': 'wallet', 'relation': 'hypernym', 'object': 'container'},
    
    # ========== VEHICLES & TRANSPORTATION ==========
    {'subject': 'aircraft', 'relation': 'hypernym', 'object': 'vehicle'},
    {'subject': 'boat', 'relation': 'hypernym', 'object': 'vehicle'},
    {'subject': 'truck', 'relation': 'hypernym', 'object': 'vehicle'},
    {'subject': 'helicopter', 'relation': 'hypernym', 'object': 'aircraft'},
    {'subject': 'sailboat', 'relation': 'hypernym', 'object': 'boat'},
    {'subject': 'canoe', 'relation': 'hypernym', 'object': 'boat'},
    {'subject': 'ship', 'relation': 'hypernym', 'object': 'boat'},
    {'subject': 'scooter', 'relation': 'hypernym', 'object': 'vehicle'},
    {'subject': 'tractor', 'relation': 'hypernym', 'object': 'vehicle'},
    {'subject': 'trailer', 'relation': 'hypernym', 'object': 'vehicle'},
    {'subject': 'van', 'relation': 'hypernym', 'object': 'vehicle'},
    
    # ========== TOOLS & UTENSILS ==========
    {'subject': 'knife', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'fork', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'scissors', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'ladder', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'broom', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'spatula', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'tong', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'ladle', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'cutter', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'chopstick', 'relation': 'hypernym', 'object': 'utensil'},
    {'subject': 'toothpick', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'cane', 'relation': 'hypernym', 'object': 'tool'},
    {'subject': 'hammer', 'relation': 'hypernym', 'object': 'tool'},
    
    # ========== CONTAINERS & VESSELS ==========
    {'subject': 'bottle', 'relation': 'hypernym', 'object': 'container'},
    {'subject': 'bucket', 'relation': 'hypernym', 'object': 'container'},
    {'subject': 'canister', 'relation': 'hypernym', 'object': 'container'},
    {'subject': 'carton', 'relation': 'hypernym', 'object': 'container'},
    {'subject': 'mug', 'relation': 'hypernym', 'object': 'cup'},
    {'subject': 'pitcher', 'relation': 'hypernym', 'object': 'container'},
    {'subject': 'platter', 'relation': 'hypernym', 'object': 'plate'},
    {'subject': 'saucer', 'relation': 'hypernym', 'object': 'plate'},
    {'subject': 'tray', 'relation': 'hypernym', 'object': 'container'},
    {'subject': 'toolbox', 'relation': 'hypernym', 'object': 'box'},
    
    # ========== FURNITURE & HOUSEHOLD ==========
    {'subject': 'bookcase', 'relation': 'hypernym', 'object': 'furniture'},
    {'subject': 'cabinet', 'relation': 'hypernym', 'object': 'furniture'},
    {'subject': 'dresser', 'relation': 'hypernym', 'object': 'furniture'},
    {'subject': 'nightstand', 'relation': 'hypernym', 'object': 'furniture'},
    {'subject': 'ottoman', 'relation': 'hypernym', 'object': 'furniture'},
    {'subject': 'bed', 'relation': 'hypernym', 'object': 'furniture'},
    {'subject': 'pillow', 'relation': 'hypernym', 'object': 'bedding'},
    {'subject': 'pillowcase', 'relation': 'hypernym', 'object': 'bedding'},
    {'subject': 'comforter', 'relation': 'hypernym', 'object': 'bedding'},
    {'subject': 'mattres', 'relation': 'hypernym', 'object': 'bedding'},
    {'subject': 'tablecloth', 'relation': 'hypernym', 'object': 'cloth'},
    {'subject': 'placemat', 'relation': 'hypernym', 'object': 'mat'},
    
    # ========== BUILDINGS & STRUCTURES ==========
    {'subject': 'barn', 'relation': 'hypernym', 'object': 'building'},
    {'subject': 'cabin', 'relation': 'hypernym', 'object': 'building'},
    {'subject': 'castle', 'relation': 'hypernym', 'object': 'building'},
    {'subject': 'garage', 'relation': 'hypernym', 'object': 'building'},
    {'subject': 'museum', 'relation': 'hypernym', 'object': 'building'},
    {'subject': 'school', 'relation': 'hypernym', 'object': 'building'},
    {'subject': 'hangar', 'relation': 'hypernym', 'object': 'building'},
    {'subject': 'office', 'relation': 'hypernym', 'object': 'building'},
    {'subject': 'pantry', 'relation': 'hypernym', 'object': 'room'},
    {'subject': 'balcony', 'relation': 'hypernym', 'object': 'room'},
    {'subject': 'hallway', 'relation': 'hypernym', 'object': 'room'},
    {'subject': 'patio', 'relation': 'hypernym', 'object': 'room'},
    {'subject': 'porch', 'relation': 'hypernym', 'object': 'room'},
    {'subject': 'ceiling', 'relation': 'hypernym', 'object': 'room'},
    {'subject': 'floor', 'relation': 'hypernym', 'object': 'room'},
    {'subject': 'wall', 'relation': 'hypernym', 'object': 'room'},
    {'subject': 'window', 'relation': 'hypernym', 'object': 'room'},
    
    # ========== APPLIANCES & ELECTRONICS ==========
    {'subject': 'dishwasher', 'relation': 'hypernym', 'object': 'appliance'},
    {'subject': 'microwave', 'relation': 'hypernym', 'object': 'appliance'},
    {'subject': 'toaster', 'relation': 'hypernym', 'object': 'appliance'},
    {'subject': 'stove', 'relation': 'hypernym', 'object': 'appliance'},
    {'subject': 'oven', 'relation': 'hypernym', 'object': 'appliance'},
    {'subject': 'camera', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'laptop', 'relation': 'hypernym', 'object': 'computer'},
    {'subject': 'monitor', 'relation': 'hypernym', 'object': 'screen'},
    {'subject': 'printer', 'relation': 'hypernym', 'object': 'machine'},
    {'subject': 'projector', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'microphone', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'speaker', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'router', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'charger', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'mouse', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'touchpad', 'relation': 'hypernym', 'object': 'device'},
    
    # ========== MATERIALS ==========
    {'subject': 'brass', 'relation': 'hypernym', 'object': 'metal'},
    {'subject': 'bronze', 'relation': 'hypernym', 'object': 'metal'},
    {'subject': 'chrome', 'relation': 'hypernym', 'object': 'metal'},
    {'subject': 'granite', 'relation': 'hypernym', 'object': 'stone'},
    {'subject': 'cotton', 'relation': 'hypernym', 'object': 'cloth'},
    {'subject': 'vinyl', 'relation': 'hypernym', 'object': 'material'},
    {'subject': 'rubber', 'relation': 'hypernym', 'object': 'material'},
    {'subject': 'foam', 'relation': 'hypernym', 'object': 'material'},
    {'subject': 'mesh', 'relation': 'hypernym', 'object': 'material'},
    
    # ========== DESCRIPTIVE ATTRIBUTES ==========
    # Physical properties
    {'subject': 'fluffy', 'relation': 'synonym', 'object': 'soft'},
    {'subject': 'puffy', 'relation': 'synonym', 'object': 'fluffy'},
    {'subject': 'furry', 'relation': 'synonym', 'object': 'hairy'},
    {'subject': 'chubby', 'relation': 'synonym', 'object': 'fat'},
    {'subject': 'skinny', 'relation': 'synonym', 'object': 'thin'},
    {'subject': 'muscular', 'relation': 'synonym', 'object': 'strong'},
    {'subject': 'sleepy', 'relation': 'synonym', 'object': 'tired'},
    {'subject': 'calm', 'relation': 'synonym', 'object': 'peaceful'},
    {'subject': 'beautiful', 'relation': 'synonym', 'object': 'pretty'},
    {'subject': 'bright', 'relation': 'synonym', 'object': 'shiny'},
    {'subject': 'colorful', 'relation': 'synonym', 'object': 'bright'},
    {'subject': 'dark', 'relation': 'synonym', 'object': 'black'},
    {'subject': 'transparent', 'relation': 'synonym', 'object': 'clear'},
    {'subject': 'translucent', 'relation': 'synonym', 'object': 'transparent'},
    {'subject': 'opaque', 'relation': 'synonym', 'object': 'solid'},
    
    # Weather/environment
    {'subject': 'cloudy', 'relation': 'synonym', 'object': 'overcast'},
    {'subject': 'cloudless', 'relation': 'synonym', 'object': 'clear'},
    {'subject': 'rainy', 'relation': 'synonym', 'object': 'wet'},
    {'subject': 'stormy', 'relation': 'synonym', 'object': 'rough'},
    {'subject': 'sunny', 'relation': 'synonym', 'object': 'bright'},
    {'subject': 'sunlit', 'relation': 'synonym', 'object': 'sunny'},
    
    # Spatial/positional
    {'subject': 'underneath', 'relation': 'synonym', 'object': 'beneath'},
    {'subject': 'atop', 'relation': 'synonym', 'object': 'on'},
    {'subject': 'beside', 'relation': 'synonym', 'object': 'next'},
    {'subject': 'outdoor', 'relation': 'synonym', 'object': 'outdoors'},
    {'subject': 'overhead', 'relation': 'synonym', 'object': 'above'},
    
    # Actions/states
    {'subject': 'bending', 'relation': 'synonym', 'object': 'bent'},
    {'subject': 'drinking', 'relation': 'synonym', 'object': 'drinking'},
    {'subject': 'cooking', 'relation': 'synonym', 'object': 'cooked'},
    {'subject': 'cleaning', 'relation': 'synonym', 'object': 'clean'},
    {'subject': 'baking', 'relation': 'synonym', 'object': 'cooked'},
    {'subject': 'fishing', 'relation': 'synonym', 'object': 'fishing'},
    {'subject': 'jumping', 'relation': 'synonym', 'object': 'jumping'},
    {'subject': 'reading', 'relation': 'synonym', 'object': 'reading'},
    {'subject': 'shopping', 'relation': 'synonym', 'object': 'shopping'},
    {'subject': 'skiing', 'relation': 'synonym', 'object': 'ski'},
    {'subject': 'surfing', 'relation': 'synonym', 'object': 'surf'},
    {'subject': 'tennis', 'relation': 'synonym', 'object': 'tennis'},
    {'subject': 'baseball', 'relation': 'synonym', 'object': 'baseball'},
    {'subject': 'soccer', 'relation': 'synonym', 'object': 'soccer'},
    
    # Size/quantity
    {'subject': 'tiny', 'relation': 'synonym', 'object': 'small'},
    {'subject': 'huge', 'relation': 'synonym', 'object': 'large'},
    {'subject': 'narrow', 'relation': 'synonym', 'object': 'thin'},
    {'subject': 'steep', 'relation': 'synonym', 'object': 'high'},
    
    # ========== PLACES & LOCATIONS ==========
    {'subject': 'beach', 'relation': 'hypernym', 'object': 'shore'},
    {'subject': 'city', 'relation': 'hypernym', 'object': 'town'},
    {'subject': 'desert', 'relation': 'hypernym', 'object': 'land'},
    {'subject': 'farm', 'relation': 'hypernym', 'object': 'land'},
    {'subject': 'garden', 'relation': 'hypernym', 'object': 'yard'},
    {'subject': 'harbor', 'relation': 'hypernym', 'object': 'port'},
    {'subject': 'meadow', 'relation': 'hypernym', 'object': 'field'},
    {'subject': 'pasture', 'relation': 'hypernym', 'object': 'field'},
    {'subject': 'river', 'relation': 'hypernym', 'object': 'water'},
    {'subject': 'zoo', 'relation': 'hypernym', 'object': 'park'},
    {'subject': 'gym', 'relation': 'hypernym', 'object': 'building'},
    {'subject': 'market', 'relation': 'hypernym', 'object': 'store'},
    {'subject': 'intersection', 'relation': 'hypernym', 'object': 'road'},
    {'subject': 'tunnel', 'relation': 'hypernym', 'object': 'road'},
    {'subject': 'bridge', 'relation': 'hypernym', 'object': 'road'},
    
    # ========== MISCELLANEOUS ==========
    {'subject': 'alarm', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'antenna', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'bell', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'clock', 'relation': 'hypernym', 'object': 'device'},
    {'subject': 'coin', 'relation': 'hypernym', 'object': 'money'},
    {'subject': 'cash', 'relation': 'hypernym', 'object': 'money'},
    {'subject': 'flag', 'relation': 'hypernym', 'object': 'symbol'},
    {'subject': 'kite', 'relation': 'hypernym', 'object': 'toy'},
    {'subject': 'balloon', 'relation': 'hypernym', 'object': 'toy'},
    {'subject': 'letter', 'relation': 'hypernym', 'object': 'mail'},
    {'subject': 'map', 'relation': 'hypernym', 'object': 'paper'},
    {'subject': 'mirror', 'relation': 'hypernym', 'object': 'glass'},
    {'subject': 'rope', 'relation': 'hypernym', 'object': 'cord'},
    {'subject': 'tape', 'relation': 'hypernym', 'object': 'material'},
    {'subject': 'towel', 'relation': 'hypernym', 'object': 'cloth'},
    {'subject': 'uniform', 'relation': 'hypernym', 'object': 'clothing'},
]

In [ ]:
FOUND = [abstract, adult, air, airplane, airport, alcohol, aluminum, ambulance, animal, apartment, appliance, aquarium, arm, armchair, 
         artichoke, artwork, asphalt, athlete, baby, backpack, bag, bagel, bakery, bald, balding, ball, bamboo, banana, bandage, bar, barren, barrier, 
         basil, basket, bat, bath, bathroom, bathtub, bead, beak, bedding, bedroom, bedspread, beet, bench, beneath, bent, berry, beverage, bicycle, bike, bird, biscuit, blank, blanket, blazer, bleacher, blender, blind, block, blond, blossom, blouse, blueberry, blurry, board, boarding, bone, book, bookshelf, bottom, 
         boulder, bowl, box, boy, bracelet, branch, bread, breakfast, 
         brick, broken, broth, brown, brunette, brush, brushing, building, bulb, bun, bunch, burger, burning, burnt, burrito, 
         bus, bush, bushy, butter, cabbage, cable, cafe, cake, calculator, calico, candle, candy, canopy, car, caramel, card, cardboard, carpet, carriage, cart, cat, catch, caucasian, cd, celery, center, cereal, chain, chair, character, cheese, cherry, child, chili, chip, chipped, chocolate, chopped, church, cigarette, classroom, clay, closet, cloth, clothe, cloud, cloudy, coach, coat, coconut, coffee, coleslaw, collar, color, comb, computer, concrete, console, container, control, controller, cookie, cooler, copper, cord, corn, costume, couch, counter, couple, cover, cracker, crate, crisp, crispy, croissant, crosswalk, crowd, crown, crystal, cucumber, cup, cupboard, cupcake, curtain, cut, daisy, deck, decoration, decorative, deep, denim, dense, desk, dessert, device, diaper, dinner, dip, dirt, dirty, dish, dispenser, display, displayed, dock, docked, dog, doll, door, doorway, dough, drape, draped, drawer, dress, drink, dry, dryer, dugout, dumpster, earphone, edge, end, engineer, entrance, envelope, exterior, extinguisher, eye, face, factory, family, fan, fancy, faucet, feather, female, fence, field, figure, figurine, finger, fire, fixture, flame, flip, flower, foam, foggy, food, foot, forest, frame, frisbee, front, fruit, fry, funny, fuzzy, gadget, garbage, garment, garnish, gas, gentleman, giant, gift, ginger, girl, glaze, glossy, glove, goal, gold, gown, graffiti, grass, gravel, gravy, gray, grazing, grill, ground, guacamole, gun, guy, hair, hamburger, hand, handbag, hang, hardwood, hat, hazy, head, headboard, headphone, heart, heater, hedge, heel, herb, herd, highway, hill, hillside, hilltop, hit, home, hoof, hook, hose, hotel, house, hung, hurdle, hydrant, ice, indoors, iron, island, jacket, jagged, jar, jean, jeans, jeep, jet, jumpsuit, kettle, keyboard, keypad, khaki, kiosk, kitchen, kiwi, knit, label, lace, lady, lake, lamp, lawn, lettuce, lid, life, light, lighthouse, line, lion, liquid, living, log, longer, lot, luggage, machine, magazine, magnet, mailbox, male, mall, man, mane, marble, marina, mask, match, material, mature, mayonnaise, meal, meat, melon, menu, metal, middle, milk, milkshake, miniature, minivan, minute, misty, mitt, mixer, money, mos, motorcycle, mound, mountain, mouth, mozzarella, mud, muddy, murky, napkin, newspaper, noodle, notebook, notepad, number, nut, oak, ocean, officer, omelette, onion, ornament, ornamental, ostrich, outdoors, outfit, outlet, package, packet, pad, paddle, paint, painting, pajamas, pan, pancake, panel, pant, paper, park, parking, parrot, parsley, past, pasta, pastry, path, pattern, pavement, paw, peel, pepperoni, person, pesto, phone, picture, pie, pineapple, pizza, plaid, plain, plant, plate, platform, player, plush, pocket, policeman, polished, pond, poodle, pool, post, poster, pot, powder, power, produce, public, pumpkin, purse, radio, railroad, raincoat, ramekin, receipt, refrigerator, remote, restaurant, restroom, rib, rice, right, ring, ripe, road, roadside, roadway, roast, robe, rock, roll, roof, rooftop, room, rope, round, rug, runway, sack, safety, sail, salad, salt, sand, sandal, sandwich, sauce, sausage, scarf, screen, sculpture, sea, seal, seat, seed, shade, shaggy, shaker, shallow, shape, sheer, sheet, shelf, shelter, shiny, shirt, shoe, shop, shorter, shower, shut, shuttle, side, sidewalk, sign, silk, simple, skateboard, skateboarder, skater, skier, skillet, skin, skirt, sky, skyscraper, smoke, smoking, smooth, snack, sneaker, snowboard, snowy, sock, soda, sofa, soup, sparse, spear, spinach, spoon, spot, spread, sprinkle, squash, stack, stadium, stainless, stair, stand, stapler, station, statue, steak, steam, steel, step, stick, stone, stool, store, straight, strawberry, stroller, stuck, styrofoam, sugar, suit, suitcase, sunflower, support, surfboard, sushi, suv, sweater, sweet, swimsuit, switch, symbol, table, tablet, tag, tail, tall, taller, tank, taxi, tea, teal, teddy, telephone, television, tent, terminal, thick, thin, thumb, tie, tile, tin, tire, tissue, toast, toddler, toilet, toiletry, tooth, toothbrush, top, topped, torn, tortilla, tower, toy, track, train, trash, trick, tv, twig, umbrella, undershirt, unripe, upside, vase, vast, vegetable, vest, video, vine, vintage, waffle, wagon, walkway, wallpaper, waste, watch, water, waterfall, watermelon, wheel, wheelchair, whisk, wicker, wild, windshield, wine, wire, wireless, woman, wood, wool, worker, wrinkly, wristband, wristwatch, yogurt
]

In [13]:
import nltk
import pandas as pd
from nltk.corpus import wordnet as wn
from collections import defaultdict
from tqdm import tqdm

# Download WordNet if not already downloaded
nltk.download('wordnet')

def format_term(term: str) -> str:
    """
    Converts terms to the required format:
    - First word lowercase
    - Subsequent words capitalized
    - No spaces or symbols between words
    
    Example:
    "toilet_bowl" -> "toiletBowl"
    "random-access memory" -> "randomAccessMemory"
    """
    # Replace common separators with spaces to normalize
    normalized = term.replace('-', ' ').replace('_', ' ').replace('/', ' ')
    
    # Split into words
    words = normalized.split()
    
    # Format: first word lowercase, rest capitalized
    if not words:
        return ""
    
    formatted = words[0].lower()
    for word in words[1:]:
        formatted += word.capitalize()
    
    return formatted

def build_vocabulary_ontology(sgg_vocab, exp_vocab, relation_types=None):
    """
    Build an ontology using WordNet relationships where both subject and object
    come from the provided vocabulary lists.
    
    Parameters:
    - sgg_vocab: List of SGG vocabulary terms
    - exp_vocab: List of experimental/reference vocabulary terms
    - relation_types: List of relationship types to extract (default: all common types)
    
    Returns:
    - pandas DataFrame with relationships
    """
    
    if relation_types is None:
        relation_types = ['hypernym', 'hyponym', 'synonym', 'part_meronym', 
                         'part_holonym', 'member_meronym', 'member_holonym',
                         'substance_meronym', 'substance_holonym']
    
    # Combine vocabularies and create a lookup set for fast membership testing
    all_vocab = set(sgg_vocab + exp_vocab)
    print(f"Total vocabulary size: {len(all_vocab)} terms")
    print(f"SGG vocab: {len(sgg_vocab)} terms")
    print(f"Exp vocab: {len(exp_vocab)} terms")
    
    relationships = []
    
    print("Extracting WordNet relationships...")
    for term in tqdm(all_vocab):
        # Clean the term for WordNet lookup
        clean_term = term.lower().replace('_', ' ').replace('-', ' ')
        
        # Get all synsets for this term
        synsets = wn.synsets(clean_term)
        
        # Use only the most common synset (first one) to avoid nonsensical relationships
        # WordNet synsets are ordered by frequency/commonness
        if not synsets:
            continue
            
        # Option 1: Use only the single most common synset
        synsets_to_process = [synsets[0]]
        
        # Option 2: Use the most common synset for each POS (uncomment if preferred)
        # synsets_to_process = []
        # pos_seen = set()
        # for synset in synsets:
        #     if synset.pos() not in pos_seen:
        #         synsets_to_process.append(synset)
        #         pos_seen.add(synset.pos())
        
        for synset in synsets_to_process:
            # Extract different types of relationships
            
            # 1. Synonyms (other lemma names in the same synset)
            if 'synonym' in relation_types:
                for lemma in synset.lemma_names():
                    lemma_clean = lemma.replace('_', ' ')
                    if lemma_clean != clean_term and lemma_clean in all_vocab:
                        relationships.append({
                            'subject': term,
                            'relation': 'synonym',
                            'object': lemma_clean
                        })
            
            # 2. Hypernyms (is-a relationships - more general)
            if 'hypernym' in relation_types:
                for hypernym in synset.hypernyms():
                    for lemma in hypernym.lemma_names():
                        lemma_clean = lemma.replace('_', ' ')
                        if lemma_clean in all_vocab:
                            relationships.append({
                                'subject': term,
                                'relation': 'hypernym',
                                'object': lemma_clean
                            })
            
            # 3. Hyponyms (is-a relationships - more specific)
            if 'hyponym' in relation_types:
                for hyponym in synset.hyponyms():
                    for lemma in hyponym.lemma_names():
                        lemma_clean = lemma.replace('_', ' ')
                        if lemma_clean in all_vocab:
                            relationships.append({
                                'subject': term,
                                'relation': 'hyponym',
                                'object': lemma_clean
                            })
            

    
    # Create DataFrame
    df = pd.DataFrame(relationships)
    
    if len(df) == 0:
        print("Warning: No relationships found!")
        return pd.DataFrame(columns=['subject', 'relation', 'object'])
    
    # Remove duplicates
    df = df.drop_duplicates()
    print(f"Found {len(df)} unique relationships")
    
    # Show relationship type distribution
    print("\nRelationship type distribution:")
    print(df['relation'].value_counts())
    
    return df

def create_formatted_ontology(sgg_vocab, exp_vocab, relation_types=None, output_path=None):
    """
    Create a formatted ontology with camelCase terms.
    """
    # Build the raw ontology
    df = build_vocabulary_ontology(sgg_vocab, exp_vocab, relation_types)
    
    if len(df) == 0:
        return df
    
    # Create formatted version
    formatted_df = df.copy()
    formatted_df['subject'] = formatted_df['subject'].apply(format_term)
    formatted_df['object'] = formatted_df['object'].apply(format_term)
    formatted_df['relation'] = formatted_df['relation'].apply(format_term)
    
    # Save if output path provided
    if output_path:
        formatted_df.to_csv(output_path, index=False)
        print(f"Saved formatted ontology to {output_path}")
    
    return formatted_df

# Example usage
if __name__ == "__main__":
    # Example vocabularies (replace with your actual lists)
    # sgg_vocab = ['person', 'car', 'tree', 'building', 'dog', 'cat', 'table', 'chair']
    # exp_vocab = ['human', 'vehicle', 'plant', 'house', 'animal', 'furniture', 'food']
    
    # Build ontology with all relationship types
    ontology = create_formatted_ontology(
        sgg_vocab, 
        exp_vocab, 
        relation_types=['hypernym', 'hyponym', 'synonym'],
        output_path='vocabulary_ontology.csv'
    )
    
    print("\nSample relationships:")
    if len(ontology) > 0:
        print(ontology.head(10))
    
    # Build ontology with only specific relationship types
    simple_ontology = create_formatted_ontology(
        sgg_vocab, 
        exp_vocab, 
        relation_types=['hypernym', 'synonym'],
        output_path='simple_vocabulary_ontology_test.csv'
    )
    
    print(f"\nSimple ontology with only hypernyms and synonyms: {len(simple_ontology)} relationships")

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/sammcmanagan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Total vocabulary size: 1720 terms
SGG vocab: 401 terms
Exp vocab: 1655 terms
Extracting WordNet relationships...


100%|██████████| 1720/1720 [00:00<00:00, 10315.14it/s]


Found 1731 unique relationships

Relationship type distribution:
relation
hyponym     897
hypernym    524
synonym     310
Name: count, dtype: int64
Saved formatted ontology to vocabulary_ontology.csv

Sample relationships:
  subject  relation   object
0   truck   hyponym      van
1   truck   hyponym  tractor
2  blouse  hypernym      top
3     oak  hypernym     wood
4  closed   synonym    close
5  closed   synonym     shut
6  closed   hyponym     seal
7  potted   synonym      pot
8  potted  hypernym    plant
9    swan   hyponym      pen
Total vocabulary size: 1720 terms
SGG vocab: 401 terms
Exp vocab: 1655 terms
Extracting WordNet relationships...


100%|██████████| 1720/1720 [00:00<00:00, 87309.42it/s]

Found 834 unique relationships

Relationship type distribution:
relation
hypernym    524
synonym     310
Name: count, dtype: int64
Saved formatted ontology to simple_vocabulary_ontology_test.csv

Simple ontology with only hypernyms and synonyms: 834 relationships


In [50]:
import pandas as pd

# Format the given terms in camelCase
given_terms = [
    "growing on", "hanging from", "has", "holding", "in", "in front of", 
    "laying on", "looking at", "lying on", "made of", "mounted on", "near",      
    "of", "on", "on back of", "over", "painted on", "parked on", "part of", 
    "playing", "riding", "says", "sitting on", "standing on", "to", "under", 
    "using", "walking in", "walking on", "watching", "wearing", "wears", "with"
]

def format_predicate(term):
    """Convert to camelCase format"""
    words = term.split()
    if not words:
        return ""
    formatted = words[0].lower()
    for word in words[1:]:
        formatted += word.capitalize()
    return formatted

# Format given terms
formatted_given = [format_predicate(term) for term in given_terms]

# Comprehensive list of common scene graph and referring expression predicates
common_predicates = [
    # Basic spatial relationships
    "on", "in", "under", "over", "above", "below", "behind", "beside", 
    "nextTo", "near", "farFrom", "between", "among", "around",
    "onTopOf", "underneath", "beneath", "beyondOf", "withinOf",
    
    # Directional spatial
    "inFrontOf", "onTopOf", "atSideOf", "toLeftOf", "toRightOf", 
    "behindOf", "aboveOf", "belowOf", "leftOf", "rightOf",
    "onLeftOf", "onRightOf", "atLeftOf", "atRightOf",
    "toTheLeftOf", "toTheRightOf", "onTheLeftOf", "onTheRightOf",
    
    # Surface contact
    "sittingOn", "standingOn", "lyingOn", "layingOn", "leaningOn", 
    "restingOn", "placedOn", "mountedOn", "attachedTo",
    "sittingOnTopOf", "standingOnTopOf", "restingOnTopOf",
    
    # Physical interactions
    "holding", "grasping", "touching", "grabbing", "carrying", "lifting",
    "pushing", "pulling", "squeezing", "hugging", "embracing",
    
    # Visual/perceptual
    "lookingAt", "watching", "seeing", "staringAt", "gazingAt", 
    "facing", "turnedToward", "observing", "viewing",
    
    # Movement/transportation
    "riding", "driving", "flying", "sailing", "walking", "running",
    "walkingOn", "walkingIn", "movingOn", "travelingOn",
    
    # Attachment/hanging
    "hangingFrom", "suspendedFrom", "danglingFrom", "swingingFrom",
    "connectedTo", "linkedTo", "joinedTo", "boundTo",
    
    # Containment
    "inside", "within", "containedIn", "enclosedBy", "surroundedBy",
    "surrounding", "enclosing", "encompassing", "wrappedIn",
    "insideOf", "withinThe", "containedWithin", "enclosedWithin",
    
    # Possession/association
    "has", "owns", "possesses", "belongsTo", "with", "accompaniedBy",
    "partOf", "componentOf", "elementOf", "memberOf",
    
    # Material/composition
    "madeOf", "composedOf", "constructedFrom", "builtFrom", "formedFrom",
    "coveredWith", "coatedWith", "filledWith", "paintedWith",
    
    # Actions
    "eating", "drinking", "reading", "writing", "playing", "sleeping",
    "working", "cooking", "cleaning", "throwing", "catching", "kicking",
    
    # Clothing/wearing
    "wearing", "wears", "dressedIn", "clothedIn", "hasOn", "sporting",
    
    # Growing/natural processes
    "growingOn", "growingIn", "bloomingOn", "attachedNaturallyTo",
    
    # Text/communication
    "says", "displays", "shows", "indicates", "reads", "spells",
    
    # Parking/stationary
    "parkedOn", "parkedAt", "parkedNear", "stationedAt", "dockedAt",
    
    # Support/foundation
    "supportedBy", "heldBy", "balancedOn", "propeedUp", "foundationOf",
    
    # Overlap/intersection
    "overlapping", "intersecting", "crossing", "spanning", "bridging",
    
    # Proximity variations
    "closeTo", "adjacent", "neighboring", "opposite", "acrossFrom",
    "nextTo", "besideOf", "alongsideOf", "closeBy", "nearBy",
    "inProximityTo", "inVicinityOf",
    
    # Attributes/properties
    "hasAttribute", "hasProperty", "hasColor", "hasSize", "hasShape",
    "hasTexture", "hasMaterial", "isColor", "isSize", "isShape",
    
    # Usage/function
    "using", "operatingWith", "controllingWith", "manipulating",
    
    # Reflection/shadow
    "reflectedIn", "shadowedBy", "mirroredIn", "castingShadowOn"
]

# Create synonym and hypernym relationships
predicate_relationships = [
    # Spatial "on" relationships
    ("sittingOn", "hypernym", "on"),
    ("standingOn", "hypernym", "on"), 
    ("lyingOn", "hypernym", "on"),
    ("layingOn", "hypernym", "on"),
    ("restingOn", "hypernym", "on"),
    ("placedOn", "hypernym", "on"),
    ("mountedOn", "hypernym", "on"),
    ("walkingOn", "hypernym", "on"),
    ("parkedOn", "hypernym", "on"),
    ("growingOn", "hypernym", "on"),
    ("paintedOn", "hypernym", "on"),
    
    # Synonyms for "on"
    ("layingOn", "synonym", "lyingOn"),
    ("restingOn", "synonym", "placedOn"),
    ("sittingOnTopOf", "hypernym", "sittingOn"),
    ("standingOnTopOf", "hypernym", "standingOn"),
    ("restingOnTopOf", "hypernym", "restingOn"),
    
    # Visual/perceptual synonyms
    ("lookingAt", "synonym", "watching"),
    ("watching", "synonym", "observing"),
    ("observing", "synonym", "viewing"),
    ("staringAt", "synonym", "gazingAt"),
    ("seeing", "hypernym", "lookingAt"),
    ("seeing", "hypernym", "watching"),
    
    # Physical interaction synonyms
    ("holding", "synonym", "grasping"),
    ("grabbing", "synonym", "grasping"),
    ("carrying", "hypernym", "holding"),
    ("hugging", "synonym", "embracing"),
    ("squeezing", "hypernym", "touching"),
    ("pushing", "hypernym", "touching"),
    ("pulling", "hypernym", "touching"),
    
    # Hanging relationships
    ("hangingFrom", "hypernym", "attachedTo"),
    ("suspendedFrom", "synonym", "hangingFrom"),
    ("danglingFrom", "synonym", "hangingFrom"),
    ("swingingFrom", "hypernym", "hangingFrom"),
    
    # Containment relationships
    ("inside", "synonym", "within"),
    ("containedIn", "synonym", "inside"),
    ("enclosedBy", "hypernym", "inside"),
    ("surroundedBy", "hypernym", "enclosedBy"),
    ("wrappedIn", "hypernym", "enclosedBy"),
    ("insideOf", "synonym", "inside"),
    ("withinThe", "synonym", "within"),
    ("containedWithin", "synonym", "containedIn"),
    ("enclosedWithin", "synonym", "enclosedBy"),
    
    # Proximity relationships
    ("nextTo", "synonym", "beside"),
    ("closeTo", "synonym", "near"),
    ("adjacent", "synonym", "nextTo"),
    ("neighboring", "synonym", "nextTo"),
    ("farFrom", "synonym", "distantFrom"),
    ("besideOf", "synonym", "beside"),
    ("alongsideOf", "synonym", "beside"),
    ("closeBy", "synonym", "closeTo"),
    ("nearBy", "synonym", "near"),
    ("inProximityTo", "hypernym", "near"),
    ("inVicinityOf", "hypernym", "near"),
    ("underneath", "synonym", "under"),
    ("beneath", "synonym", "under"),
    ("withinOf", "synonym", "within"),
    
    # Directional relationships
    ("inFrontOf", "hypernym", "near"),
    ("behind", "hypernym", "near"),
    ("toLeftOf", "hypernym", "beside"),
    ("toRightOf", "hypernym", "beside"),
    ("leftOf", "synonym", "toLeftOf"),
    ("rightOf", "synonym", "toRightOf"),
    ("onLeftOf", "synonym", "leftOf"),
    ("onRightOf", "synonym", "rightOf"),
    ("atLeftOf", "synonym", "leftOf"),
    ("atRightOf", "synonym", "rightOf"),
    ("toTheLeftOf", "synonym", "toLeftOf"),
    ("toTheRightOf", "synonym", "toRightOf"),
    ("onTheLeftOf", "synonym", "onLeftOf"),
    ("onTheRightOf", "synonym", "onRightOf"),
    ("aboveOf", "synonym", "above"),
    ("belowOf", "synonym", "below"),
    ("onTopOf", "hypernym", "above"),
    ("onBackOf", "hypernym", "on"),
    ("beyondOf", "hypernym", "behind"),
    
    # Possession/association
    ("owns", "synonym", "possesses"),
    ("belongsTo", "hypernym", "with"),
    ("accompaniedBy", "synonym", "with"),
    ("hasOn", "synonym", "wearing"),
    ("wears", "synonym", "wearing"),
    ("isWearing", "synonym", "wearing"),
    ("dressedIn", "hypernym", "wearing"),
    ("clothedIn", "hypernym", "wearing"),
    ("sporting", "synonym", "wearing"),
    
    # Part/component relationships
    ("partOf", "synonym", "componentOf"),
    ("elementOf", "synonym", "componentOf"),
    ("memberOf", "hypernym", "partOf"),
    
    # Material/composition
    ("composedOf", "synonym", "madeOf"),
    ("constructedFrom", "hypernym", "madeOf"),
    ("builtFrom", "hypernym", "madeOf"),
    ("formedFrom", "hypernym", "madeOf"),
    ("coveredWith", "hypernym", "has"),
    ("coatedWith", "synonym", "coveredWith"),
    ("filledWith", "hypernym", "has"),
    ("paintedWith", "hypernym", "coveredWith"),
    
    # Connection relationships
    ("connectedTo", "synonym", "linkedTo"),
    ("joinedTo", "synonym", "linkedTo"),
    ("boundTo", "hypernym", "attachedTo"),
    ("attachedTo", "hypernym", "connectedTo"),
    
    # Support relationships
    ("supportedBy", "hypernym", "on"),
    ("heldBy", "hypernym", "supportedBy"),
    ("balancedOn", "hypernym", "on"),
    ("foundationOf", "hypernym", "supportedBy"),
    
    # Movement/transportation
    ("driving", "hypernym", "riding"),
    ("flying", "hypernym", "riding"),
    ("sailing", "hypernym", "riding"),
    ("travelingOn", "hypernym", "movingOn"),
    ("walkingIn", "hypernym", "walking"),
    ("walkingOn", "hypernym", "walking"),
    
    # Actions
    ("reading", "hypernym", "lookingAt"),
    ("writing", "hypernym", "using"),
    ("cooking", "hypernym", "using"),
    ("cleaning", "hypernym", "using"),
    ("throwing", "hypernym", "using"),
    ("catching", "hypernym", "using"),
    ("kicking", "hypernym", "using"),
    ("playing", "hypernym", "using"),
    
    # Parking/stationary
    ("parkedAt", "hypernym", "parkedOn"),
    ("parkedNear", "hypernym", "near"),
    ("stationedAt", "synonym", "parkedAt"),
    ("dockedAt", "hypernym", "parkedAt"),
    
    # Natural processes
    ("growingIn", "hypernym", "in"),
    ("bloomingOn", "hypernym", "growingOn"),
    ("attachedNaturallyTo", "hypernym", "attachedTo"),
    
    # Usage/operation
    ("operatingWith", "hypernym", "using"),
    ("controllingWith", "hypernym", "using"),
    ("manipulating", "hypernym", "using"),
    
    # Over relationships
    ("above", "hypernym", "over"),
    ("spanning", "hypernym", "over"),
    ("bridging", "hypernym", "over"),
    ("crossing", "hypernym", "over"),
    
    # Under relationships  
    ("below", "hypernym", "under"),
    ("beneath", "synonym", "under"),
    
    # Text/communication
    ("displays", "synonym", "shows"),
    ("indicates", "hypernym", "shows"),
    ("reads", "hypernym", "says"),
    ("spells", "hypernym", "says"),
    
    # Reflection/visual effects
    ("mirroredIn", "synonym", "reflectedIn"),
    ("shadowedBy", "hypernym", "near"),
    ("castingShadowOn", "hypernym", "over"),
    
    # Additional spatial synonyms
    ("around", "hypernym", "near"),
    ("encompassing", "hypernym", "surrounding"),
    ("overlapping", "hypernym", "intersecting"),
    ("opposite", "hypernym", "acrossFrom"),
    
    # Facing relationships
    ("facing", "hypernym", "lookingAt"),
    ("turnedToward", "hypernym", "facing"),
    
    # Attribute relationships
    ("hasColor", "hypernym", "hasAttribute"),
    ("hasSize", "hypernym", "hasAttribute"),
    ("hasShape", "hypernym", "hasAttribute"),
    ("hasTexture", "hypernym", "hasAttribute"),
    ("hasMaterial", "hypernym", "hasAttribute"),
    ("hasProperty", "synonym", "hasAttribute"),
    ("isColor", "hypernym", "hasAttribute"),
    ("isSize", "hypernym", "hasAttribute"),
    ("isShape", "hypernym", "hasAttribute"),
    ("isTexture", "hypernym", "hasAttribute"),
    ("isMaterial", "hypernym", "hasAttribute"),
    ("has", "synonym", "hasAttribute"),
    
    
    ("hangingFrom", "hypernym", "from"),
    ("holding", "synonym", "gripping"),
    ("holding", "hypernym", "touching"),
    ("madeOf", "synonym", "formedFrom"),
    ("partOf", "synonym", "elementOf"),
    ("partOf", "hypernym", "belongsTo"),
    ("playing", "hypernym", "interactingWith"),
    ("riding", "synonym", "mountedOn"),
    ("using", "synonym", "utilizing"),
    ("using", "synonym", "employing"),
    ("using", "hypernym", "interactingWith"),
    ("mountedOn", "hypernym", "placedOn"),
    ("parkedOn", "hypernym", "stationedOn"),
    ("over", "synonym", "atop"),
    ("under", "synonym", "beneath"),
    ("has", "synonym", "contains"),
    ("walkingOn", "hypernym", "walking")
]

def create_predicate_ontology():
    """Create a comprehensive predicate ontology DataFrame"""
    
    # Combine all predicates
    all_predicates = list(set(formatted_given + common_predicates))
    
    # Create DataFrame from relationships
    df = pd.DataFrame(predicate_relationships, columns=['subject', 'relation', 'object'])
    
    print(f"Created ontology with:")
    print(f"  {len(all_predicates)} unique predicates")
    print(f"  {len(df)} relationships")
    print(f"  Relationship types: {df['relation'].value_counts().to_dict()}")
    

    
    return df, all_predicates

def save_predicate_ontology(output_path="predicate_ontology.csv"):
    """Save the predicate ontology to CSV"""
    df, predicates = create_predicate_ontology()
    df.to_csv(output_path, index=False)
    
    # Also save the full predicate list
    predicate_df = pd.DataFrame({'predicate': sorted(predicates)})
    predicate_df.to_csv(output_path.replace('.csv', '_vocabulary.csv'), index=False)
    
    print(f"Saved ontology to {output_path}")
    print(f"Saved vocabulary to {output_path.replace('.csv', '_vocabulary.csv')}")
    
    return df

# Example usage
if __name__ == "__main__":
    # Create and save the ontology
    ontology_df = save_predicate_ontology()
    
    # Show some examples
    print("\nSample relationships:")
    print(ontology_df.head(15))
    
    print("\nRelationship distribution:")
    print(ontology_df['relation'].value_counts())
    
    print(f"\nFormatted given terms:")
    for original, formatted in zip(given_terms[:10], formatted_given[:10]):
        print(f"  '{original}' -> '{formatted}'")

Created ontology with:
  194 unique predicates
  175 relationships
  Relationship types: {'hypernym': 107, 'synonym': 68}
Saved ontology to predicate_ontology.csv
Saved vocabulary to predicate_ontology_vocabulary.csv

Sample relationships:
            subject  relation      object
0         sittingOn  hypernym          on
1        standingOn  hypernym          on
2           lyingOn  hypernym          on
3          layingOn  hypernym          on
4         restingOn  hypernym          on
5          placedOn  hypernym          on
6         mountedOn  hypernym          on
7         walkingOn  hypernym          on
8          parkedOn  hypernym          on
9         growingOn  hypernym          on
10        paintedOn  hypernym          on
11         layingOn   synonym     lyingOn
12        restingOn   synonym    placedOn
13   sittingOnTopOf  hypernym   sittingOn
14  standingOnTopOf  hypernym  standingOn

Relationship distribution:
relation
hypernym    107
synonym      68
Name: count, dtype:

In [53]:
full_ontology = pd.concat([simple_ontology, ontology_df], ignore_index=True)
print(f"\nCombined ontology size: {len(full_ontology)} relationships")
full_ontology.to_csv('full_vocabulary_ontology.csv', index=False)
print(len(simple_ontology))


Combined ontology size: 981 relationships
806


In [ ]:
import pandas as pd
from tqdm import tqdm

def create_ontology_vocabulary(ontology_csv_path):
    """
    Extract all unique terms from the ontology CSV to create a vocabulary set.
    
    Parameters:
    - ontology_csv_path: Path to the ontology CSV file
    
    Returns:
    - Set of all unique terms from subject, relation, and object columns
    """
    print(f"Loading ontology from {ontology_csv_path}...")
    ontology_df = pd.read_csv(ontology_csv_path)
    
    print(f"Ontology contains {len(ontology_df)} relationships")
    print("Columns:", ontology_df.columns.tolist())
    
    # Extract all unique terms from subject, relation, and object columns
    vocabulary = set()

    # Add subjects
    subjects = ontology_df['subject'].dropna().unique()
    vocabulary.update(subjects)
    print(f"Added {len(subjects)} unique subjects")
    
    
    # Add objects
    objects = ontology_df['object'].dropna().unique()
    vocabulary.update(objects)
    print(f"Added {len(objects)} unique objects")
    
    print(f"Total ontology vocabulary size: {len(vocabulary)} terms")
    
    return vocabulary

def filter_scene_graph_by_vocabulary(scene_graph_csv_path, vocabulary, output_path=None, chunk_size=10000):
    """
    Filter scene graph CSV to keep only rows where subject, relation, and object
    are all present in the ontology vocabulary.
    
    Parameters:
    - scene_graph_csv_path: Path to the scene graph CSV file
    - vocabulary: Set of allowed terms from ontology
    - output_path: Path to save filtered CSV (optional)
    - chunk_size: Process file in chunks for memory efficiency
    
    Returns:
    - Filtered pandas DataFrame
    """
    print(f"Loading scene graph from {scene_graph_csv_path}...")
    
    # Check file size first
    try:
        sg_df = pd.read_csv(scene_graph_csv_path, nrows=5)
        print("Columns in scene graph:", sg_df.columns.tolist())
        print("Sample data:")
        print(sg_df.head())
    except Exception as e:
        print(f"Error reading file: {e}")
        return None
    
    # Process in chunks for memory efficiency with large files
    filtered_chunks = []
    total_rows = 0
    kept_rows = 0
    
    print("Filtering scene graph...")
    
    for chunk in tqdm(pd.read_csv(scene_graph_csv_path, chunksize=chunk_size)):
        total_rows += len(chunk)
        
        # Filter rows where all three columns are in vocabulary
        mask = (
            chunk['subject'].isin(vocabulary) & 
            chunk['relationship'].isin(vocabulary) & 
            chunk['object'].isin(vocabulary)
        )
        
        filtered_chunk = chunk[mask]
        kept_rows += len(filtered_chunk)
        
        if len(filtered_chunk) > 0:
            filtered_chunks.append(filtered_chunk)
    
    if not filtered_chunks:
        print("Warning: No rows passed the vocabulary filter!")
        return pd.DataFrame(columns=['subject', 'relation', 'object'])
    
    # Combine all filtered chunks
    filtered_df = pd.concat(filtered_chunks, ignore_index=True)
    
    print(f"Filtering complete:")
    print(f"  Original rows: {total_rows:,}")
    print(f"  Filtered rows: {kept_rows:,}")
    print(f"  Retention rate: {(kept_rows/total_rows)*100:.2f}%")
    
    # Show some statistics
    print(f"\nFiltered data statistics:")
    print(f"  Unique subjects: {filtered_df['subject'].nunique()}")
    print(f"  Unique relations: {filtered_df['relationship'].nunique()}")
    print(f"  Unique objects: {filtered_df['object'].nunique()}")
    
    # Save if output path provided
    if output_path:
        filtered_df.to_csv(output_path, index=False)
        print(f"Saved filtered scene graph to {output_path}")
    
    return filtered_df

def show_filtering_analysis(scene_graph_csv_path, vocabulary, sample_size=10000):
    """
    Analyze what gets filtered out to help debug vocabulary issues.
    """
    print("Analyzing filtering impact...")
    
    # Load a sample of the data
    sample_df = pd.read_csv(scene_graph_csv_path, nrows=sample_size)
    
    # Check each column separately
    subject_in_vocab = sample_df['subject'].isin(vocabulary)
    relation_in_vocab = sample_df['relationship'].isin(vocabulary)
    object_in_vocab = sample_df['object'].isin(vocabulary)
    
    print(f"\nVocabulary coverage in sample of {len(sample_df)} rows:")
    print(f"  Subjects in vocabulary: {subject_in_vocab.sum()}/{len(sample_df)} ({(subject_in_vocab.sum()/len(sample_df))*100:.1f}%)")
    print(f"  Relations in vocabulary: {relation_in_vocab.sum()}/{len(sample_df)} ({(relation_in_vocab.sum()/len(sample_df))*100:.1f}%)")
    print(f"  Objects in vocabulary: {object_in_vocab.sum()}/{len(sample_df)} ({(object_in_vocab.sum()/len(sample_df))*100:.1f}%)")
    
    # Show examples of terms not in vocabulary
    print(f"\nExamples of terms NOT in vocabulary:")
    
    missing_subjects = sample_df[~subject_in_vocab]['subject'].unique()[:10]
    missing_relations = sample_df[~relation_in_vocab]['relationship'].unique()[:10]
    missing_objects = sample_df[~object_in_vocab]['object'].unique()[:10]
    
    print(f"  Missing subjects: {list(missing_subjects)}")
    print(f"  Missing relations: {list(missing_relations)}")
    print(f"  Missing objects: {list(missing_objects)}")

# Main execution function
def main(ontology_csv_path, scene_graph_csv_path, output_path=None, analyze_first=True):
    """
    Complete workflow to filter scene graph using ontology vocabulary.
    """
    # Step 1: Create vocabulary from ontology
    vocabulary = create_ontology_vocabulary(ontology_csv_path)
    
    # Step 2: Analyze impact (optional)
    if analyze_first:
        show_filtering_analysis(scene_graph_csv_path, vocabulary)
        
        response = input("\nProceed with filtering? (y/n): ")
        if response.lower() != 'y':
            print("Filtering cancelled.")
            return None
    
    # Step 3: Filter the scene graph
    filtered_df = filter_scene_graph_by_vocabulary(
        scene_graph_csv_path, 
        vocabulary, 
        output_path
    )
    print(f"\nFiltered dataset has {len(filtered_df)} relationships.")
    df_filtered = filtered_df[filtered_df['image_id'].isin(filtered_df['image_id'].value_counts()[filtered_df['image_id'].value_counts() >= 10].index)]
    print(f"After filtering by image_id, dataset has {len(df_filtered)} relationships.")
    return df_filtered

# Example usage
if __name__ == "__main__":
    # Replace with your actual file paths
    ontology_path = "full_vocabulary_ontology.csv"
    scene_graph_path = "/Users/sammcmanagan/Desktop/Thesis/Model/data/vg/vg_scene_graphs_formatted.csv"
    output_path = "vocab_filtered_vg_scene_graph.csv"
    
    # Run the filtering
    filtered_data = main(
        ontology_csv_path=ontology_path,
        scene_graph_csv_path=scene_graph_path,
        output_path=output_path,
        analyze_first=True  # Set to False to skip analysis
    )
    
    if filtered_data is not None:
        print(f"\nFiltering complete! Final dataset has {len(filtered_data)} relationships.")
        print("\nSample of filtered data:")
        print(filtered_data.head(10))